# Welcome to the GEP Generator

This Jupyter based interface has been designed to support scenario runs for the Global Electrification Platform. 

The interface is built upon a modified version of [OnSSET](http://www.onsset.org/) developed by KTH dESA to provide an easy and quick way to generate electrification investment scenarios compatible with GEP data guidelines and protocols. 

Follow the steps below to generate custom electrification investment outlooks for your country of interest.

#### Start by importing the code 

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), 'gep_onsset'))
import os, json
from onsset import *
from IPython.display import display, Markdown, HTML
%matplotlib inline
from guinea_demand_customization import apply_customized_demand

**Note**: In case you encounter an error please refer to the import section of the onsset.py code.

# 1. GIS data selection

First, run the cell below to browse to the directory your input CSV file is located at and select the input file. Sample file shall be located at .\ gep-onsset\test_data. 


In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
from openpyxl import load_workbook
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
messagebox.showinfo('OnSSET', 'Open the input file with extracted GIS data')
input_file = filedialog.askopenfilename()

onsseter = SettlementProcessor(input_file)

In [ ]:
pv_path = os.path.abspath(os.path.join(r'GIN Data\Solar profile', 'gn-2-pv.csv'))
wind_path = os.path.abspath(os.path.join(r'GIN Data\Wind profile', 'gn-2-wind.csv'))

# 2. Modelling period and target electrification rate

Next, define the modelling period and the electrification rate to be achieved by the end of the analysis, and also for an intermediate year. Further down you will also define an intermediate year and target (in the ***Levers*** section).

*On the GEP Explorer, the end year electrification rate target is set to 100%, and the intermediate target is set to follow a linear increase from the current level. If a country for example has 52% access by 2020, the target would be to increase electrification rate by ~4% per year to reach 100% by 2030. Therefore the intermediate target would be 80% by 2025.*

In [ ]:
start_year = 2025
end_year = 2040
electrification_rate_target = 1 # E.g. 1 for 100% electrification rate or 0.80 for 80% electrification rate

intermediate_year = 2030
intermediate_electrification_target = 0.7 # E.g. for a target electrification rate of 75%, enter 0.75

yearsofanalysis = [intermediate_year, end_year]
eleclimits = {intermediate_year: intermediate_electrification_target, end_year: electrification_rate_target}
time_steps = {intermediate_year: intermediate_year-start_year, end_year: end_year-intermediate_year}

# 3. Levers

Next, define the values of the levers. These are the 6 levers that are available on the GEP Explorer. Contrary to the GEP Explorer where each lever has two or three pre-defined values, here they can take any value.

#### Lever 1: Population growth

For the first lever first, enter the expected population in the country by the end year of the analysis (e.g. 2030).

*The default values in the GEP Explorer are based on the medium growth variant and high growth variant of the UN Population Database, found <a href="https://population.un.org/wpp/" target="_blank">here</a>.*

#### Lever 2: Electricity demand target
For the second lever, enter the target tier (level of electricity access) for urban and rural households respectively. This can take a value between "1" (lowest level of electricity access) and "5" (highest level of electricity access) as in ESMAPs Multi-Tier Framework for Measuring Electricity Access (found <a href="https://www.esmap.org/node/55526" target="_blank">here</a>). Alternatively, enter "6" to use a distribution of the tiers across the country based on poverty levels and GDP according to the methodology found <a href="https://www.mdpi.com/1996-1073/12/7/1395" target="_blank">here</a>.

*On the GEP Explorer, the following three electricity demand targets are used:*

*Top down demand target - Low: In this case, all urban clusters are tergeted to reach the current average consumption level of electrified households, and rural settlements are assigned Tier 1.*

*Top down demand target - High: In this case, all urban clusters are tergeted to reach one Tier higher than the current average consumption level of electrified households, and rural settlements are assigned Tier 3.*

*Bottom up demand target (Poverty - GDP): In this case each settlement is assigned a demand target based on poverty and GDP levels as described in the methodology above. Choose "6" for both the urban_target_tier and rural_target_tier to use this option*

In [ ]:
end_year_pop = 27697655

In [ ]:
urban_target_tier = 6
rural_target_tier = 6

#### Lever 3: Intermediate investment plan
For the third lever, restrictions can be placed on how many grid connections can be achieved in a year.

*On the GEP Explorer there are two options that explore how to achieve the intermediate electrification target. Not capped scenario assumes that the electrification rate increases linearly over the modelling years; no restrictions are set in terms of feasible grid connections per year. Capped growth scenario assumes that the electrification rate increases as in the first scenario but is subject to grid connections limitation, equal to 2.5% of population per year. Note that on the platfrom, there are no restrictions applied after 2025, thus the restriction for 2030 is always set to "999999999"*

In [ ]:
# This is the maximum amount of new households that can be connected to the grid in one year (thousands)
annual_new_grid_connections_limit_intermediate = 127  #999999999 # Limit per year from start year (e.g. 2018) to intermediate year (e.g. 2025)
annual_new_grid_connections_limit_end = 999999999 # Limit per year from intermediate year (e.g. 2025) to end year (e.g. 2030)

#### Lever 4: Grid generating cost of electricity
This lever examines different average costs of generating electricity by the power-plants connected to the national grid. This cost is one of the factors that affect the LCoE of connecting to the grid (together with extension of the grid lines etc.), and may affect the split between grid- and off-grid technologies.

*On the GEP Explorer, the Estimated on-grid cost is based on either national plans or a power optimization model. The High-on grid cost is assumed to be 25% higher*

In [ ]:
grid_generation_cost = 0.22 ### This is the grid cost electricity USD/kWh as expected in the end year of the analysis

#### Lever 5: PV system cost adjustment
This lever reflects the role of PV system costs on electrification results. All PV based systems will be adjusted by a factor to simulate a higher or lower cost of PV systems (compared to the baseline values entered below). A value lower than 1 means lower investment costs for PV systems compared to baseline projection, and a value larger than 1 means higher investment cost for PV systems compared to baseline. E.g. "0.75" would mean a cost that is 25% lower compared to baseline costs.

*On the GEP Explorer, three values for this parameter are explored:*

*Expected PV cost: Adjustment factor set to 1*

*High PV cost: Adjustment factor set to 1.25 to simulate 25% higher costs*

*Low PV cost: Adjustment factor set to 0.75 to simulate 25% lower cost*

In [ ]:
pv_adjustment_factor = 1

#### Lever 6: Rollout plan
This lever reflects the electrification approach to be examined. On the GEP Explorer, there are currently two options in use:

**Nationwide Least Cost approach:** This options aims to achieve the electrification rate targets set for the intermediate and end year. For the years where the target is set below 100%, the algorithm prioritizes grid densification first (ramp up in already electrified clusters) then selection based on lowest invetsment cost per capita to choose which clusters to be electrified. Note! a generation capacity limit for the centralized grid is applied in order to keep grid expansion to sensible levels until the intermediate year. On the GEP Explorer, this follows historical trends for the countries in the region that managed the largest relative increase in recent years.*

**Forced grid approach:** This can take the following forms.

- Forced grid under a defined buffer zone (auto_intensification set equal to Xkm) & least cost approach outside of the buffer zone.

- Forced grid under a defined buffer zone (auto_intensification set equal to Xkm) & no electrification outside the buffer zone.

In [ ]:
prioritization = 5                        # Select 5, 4 or 6; 5 = Nationwide Least Cost approach, 4 = Forced grid connection within a buffer only, 6 = Forced grid within buffer + least cost out of the buffer

auto_intensification = 0                 # Buffer distance (km) for automatic intensification if choosing prioritization 4 or 6

# This is the maximum generation capacity that can be added to the grid in one year (MW)
annual_grid_cap_gen_limit_intermediate = 90 # Limit per year from start year (e.g. 2018) to intermediate year (e.g. 2025)
annual_grid_cap_gen_limit_end = 999999999 # Limit per year from intermediate year (e.g. 2025) to end year (e.g. 2030)

# 4. Enter country specific data

In addition to the levers above the user can customize a large number of variables describing the social - economic - technological environment in the selected country. 

**Note!** Most input values shall represent future estimates for the variable, i.e. they describe and **NOT** current values.

### a. Demographics and Social components

In [ ]:
pop_start_year = 17521167       ### Population in the base year (2025) — RGPH-4 census, confirmed

urban_ratio_start_year = 0.45  ### Urban population ratio in the base year (2025) — RGPH-4: ~38.7% urban, ~61.3% rural
urban_ratio_end_year = 0.5     ### Urban population ratio in the end year — placeholder, adjust with a real projection (see note below)

num_people_per_hh_urban = 6.6
num_people_per_hh_rural = 6.3

elec_ratio_start_year = 0.538
urban_elec_ratio = 0.91
rural_elec_ratio = 0.23

### b. Technology specifications & costs

The cell below contains all the information that is used to calculate the levelized costs for all the technologies, including grid. These default values should be updated to reflect the most accurate values in the country. There are currently 7 potential technologies to include in the model:
* Grid
* PV Mini-grid
* Wind Mini-grid
* Hydro Mini-grid
* Diesel Mini-grid
* PV Stand-alone systems
* Diesel Stand-alone systems

First, decide whether to include diesel technologies or not:

In [ ]:
diesel_techs = 0
tech_switches = {'sa_pv': 1, 'mg_hydro': 1, 'mg_pv': 0, 'mg_pv_hybrid': 1, 'mg_wind_hybrid': 0}

In [ ]:
grid_power_plants_capital_cost = 1363   ### The cost in USD/kW is for capacity upgrades of the grid
grid_losses = 0.18                     ### The fraction of electricity lost in transmission and distribution (percentage)  

In [ ]:
diesel_price = 0.5                     ### This is the diesel price in USD/liter as expected in the end year of the analysis

sa_diesel_cost = {'diesel_price': diesel_price,
                  'efficiency': 0.28,
                  'diesel_truck_consumption': 14,
                  'diesel_truck_volume': 300}

mg_diesel_cost = {'diesel_price': diesel_price,
                  'efficiency': 0.33,
                  'diesel_truck_consumption': 14,
                  'diesel_truck_volume': 300}

In [ ]:
sa_diesel_capital_cost = {float("inf"): 938}      ### Stand-alone Diesel capital cost (USD/kW) as expected in the years of the analysis
mg_diesel_capital_cost = {float("inf"): 721}      ### Mini-grid Diesel capital cost (USD/kW) as expected in the years of the analysis
mg_pv_capital_cost = {float("inf"): 2950}         ### Mini-grid PV capital cost (USD/kW) as expected in the years of the analysis
mg_wind_capital_cost = {float("inf"): 3750}       ### Mini-grid Wind capital cost (USD/kW) as expected in the years of the analysis
mg_hydro_capital_cost = {float("inf"): 3000}      ### Mini-grid Hydro capital cost (USD/kW) as expected in the years of the analysis

In [ ]:
sa_pv_capital_cost_1 = 9620          ### Stand-alone PV capital cost (USD/kW) for household systems under 20 W
sa_pv_capital_cost_2 = 8780          ### Stand-alone PV capital cost (USD/kW) for household systems between 21-50 W
sa_pv_capital_cost_3 = 6380           ### Stand-alone PV capital cost (USD/kW) for household systems between 51-100 W
sa_pv_capital_cost_4 = 4470           ### Stand-alone PV capital cost (USD/kW) for household systems between 101-1000 W
sa_pv_capital_cost_5 = 6950           ### Stand-alone PV capital cost (USD/kW) for household systems over 1 kW

The cells below contain additional technology specifications

In [ ]:
discount_rate = 0.08 # E.g. 0.08 means a discount rate of 8%
existing_grid_cost_ratio = 0.1        ### The additional cost per round of electrification (percentage) 

# Transmission and distribution costs
hv_line_capacity=69 # kV
hv_line_cost=53000 # USD/km
mv_line_cost = 7000 # USD/kW
mv_line_capacity=33 # kV
mv_line_max_length=50 # km
mv_increase_rate=0.1
max_mv_line_dist = 50 # km
MV_line_amperage_limit = 8  # Ampere (A)
lv_line_capacity=0.24 #kV
lv_line_max_length=0.5 # km
lv_line_cost=4250 # USD/km
service_Transf_type=50  # kVa
service_Transf_cost=4250  # $/unit
max_nodes_per_serv_trans=300  # maximum number of nodes served by each service transformer
hv_lv_transformer_cost=25000 # USD/unit
hv_mv_transformer_cost=25000 # USD/unit
mv_lv_transformer_cost=10000 # USD/unit
mv_mv_transformer_cost=10000 # USD/unit


# Centralized grid costs
grid_calc = Technology(om_of_td_lines=0.02,
                        distribution_losses=grid_losses,
                        connection_cost_per_hh=125,
                        base_to_peak_load_ratio=0.8,
                        capacity_factor=1,
                        tech_life=30,
                        grid_penalty_ratio=1,
                        grid_capacity_investment=grid_power_plants_capital_cost,
                        grid_price=grid_generation_cost)

mg_pv_hybrid_calc = Technology(om_of_td_lines=0.02,
                               distribution_losses=0.05,
                               connection_cost_per_hh=100,
                               capacity_factor=0.5,
                               tech_life=20,
                               mini_grid=True,
                               hybrid=True)

mg_wind_hybrid_calc = Technology(om_of_td_lines=0.02,
                                 distribution_losses=0.05,
                                 connection_cost_per_hh=100,
                                 capacity_factor=0.5,
                                 tech_life=20,
                                 mini_grid=True,
                                 hybrid=True)

# Mini-grid hydro costs
mg_hydro_calc = Technology(om_of_td_lines=0.02,
                            distribution_losses=0.05,
                            connection_cost_per_hh=100,
                            base_to_peak_load_ratio=0.85,
                            capacity_factor=0.5,
                            tech_life=30,
                            capital_cost=mg_hydro_capital_cost,
                            om_costs=0.02,
                            )

# Mini-grid wind costs
mg_wind_calc = Technology(om_of_td_lines=0.02,
                            distribution_losses=0.05,
                            connection_cost_per_hh=100,
                            base_to_peak_load_ratio=0.85,
                            capital_cost=mg_wind_capital_cost,
                            om_costs=0.02,
                            tech_life=20,
                            )

# Mini-grid PV costs
mg_pv_calc = Technology(om_of_td_lines=0.02,
                        distribution_losses=0.05,
                        connection_cost_per_hh=100,
                        base_to_peak_load_ratio=0.85,
                        tech_life=20,
                        om_costs=0.015,
                        capital_cost= mg_pv_capital_cost                        
                       )

# Stand-alone PV costs
sa_pv_calc = Technology(base_to_peak_load_ratio=0.9,
                        tech_life=15,
                        om_costs=0.02,
                        capital_cost={0.020: sa_pv_capital_cost_1 * pv_adjustment_factor, 
                                      0.050: sa_pv_capital_cost_2 * pv_adjustment_factor, 
                                      0.100: sa_pv_capital_cost_3 * pv_adjustment_factor, 
                                      1: sa_pv_capital_cost_4 * pv_adjustment_factor, 
                                      5: sa_pv_capital_cost_5 * pv_adjustment_factor},
                        standalone=True
                        )

# Mini-grid diesel costs
mg_diesel_calc = Technology(om_of_td_lines=0.03,
                            distribution_losses=0.05,
                            connection_cost_per_hh=100,
                            base_to_peak_load_ratio=0.85,
                            capacity_factor=0.7,
                            tech_life=15,
                            om_costs=0.1,
                            efficiency=0.33,
                            capital_cost=mg_diesel_capital_cost,
                            diesel_price=diesel_price,
                            diesel_truck_consumption=33.7,
                            diesel_truck_volume=15000,
                            )

# Stand-alone diesel costs
sa_diesel_calc = Technology(base_to_peak_load_ratio=0.9,
                            capacity_factor=0.7,
                            tech_life=10,
                            om_costs=0.1,
                            capital_cost=sa_diesel_capital_cost,
                            diesel_price=diesel_price,
                            standalone=True,
                            efficiency=0.28,
                            diesel_truck_consumption=14,
                            diesel_truck_volume=300)

# 5. Calibration of start year values and general information

The following steps calibrate the start year conditions in the country in terms of population and current electrification rate, and also adds some additional useful information to be used in the further calculations. If you have retrieved the input file from Energydata.info some of these steps are already completed, and you may skip those cells below where there is a note. 

In [ ]:
pop_modelled, urban_modelled = onsseter.calibrate_current_pop_and_urban(pop_start_year, urban_ratio_start_year)

onsseter.project_pop_and_urban(pop_modelled, end_year_pop, urban_modelled, urban_ratio_end_year, start_year, end_year, intermediate_year)

display(Markdown('#### The csv file has been imported correctly. Here is a preview:'))
display(onsseter.df[['Country','Pop','NightLights','TravelHours','GHI','WindVel','Hydropower','HydropowerDist']].sample(7))

**Note!**
*If you have retrieved an input file from Energydata.info, this step has already been completed and does not need to be run again. You may proceed to the next step*

In [ ]:
onsseter.df['PerCapitaDemand'] = 0.0
onsseter.df['ElectrificationOrder'] = 0.0
onsseter.df['Conflict'] = 0.0

In [ ]:
onsseter.condition_df()
onsseter.df['GridPenalty'] = onsseter.grid_penalties(onsseter.df)
onsseter.df['WindCF'] = onsseter.calc_wind_cfs()

#### Calibration of currently electrified settlements

The model calibrates which settlements are likely to be electrified in the start year, to match the national statistical values defined above. A settlement is considered to be electrified if it meets all of the following conditions:
- Has more night-time lights than the defined threshold (this is set to 0 by default)
- Is closer to the existing grid network than the distance limit
- Has more population than the threshold

First, define the threshold limits. Then run the calibration and check if the results seem okay. Else, redefine these thresholds and run again.

**Note!**
*If you have retrieved an input file from Energydata.info, this step has already been completed and does not need to be run again. You may proceed to the next step*

In [ ]:
min_night_lights = 0    ### 0 Indicates no night light, while any number above refers to the night-lights intensity
min_pop = 50      ### Settlement population above which we can assume that it could be electrified

max_service_transformer_distance = 2    ### Distance  in km from the existing grid network below which we can assume a settlement could be electrified
max_mv_line_distance = 2
max_hv_line_distance = 5

elec_modelled, urban_internal_elec_ratio, rural_internal_elec_ratio = onsseter.elec_current_and_future(elec_ratio_start_year, 
                                                                                                       urban_elec_ratio, 
                                                                                                       rural_elec_ratio, 
                                                                                                       start_year,
                                                                                                       min_night_lights=min_night_lights, 
                                                                                                       min_pop=min_pop, 
                                                                                                       max_transformer_dist=max_service_transformer_distance, 
                                                                                                       max_mv_dist=max_mv_line_distance, 
                                                                                                       max_hv_dist=max_hv_line_distance) 


In [ ]:


os.makedirs('Calibrated', exist_ok=True)

thr = globals().get('pop_threshold', 0)
calibrated = onsseter.df.loc[onsseter.df[SET_POP] > thr]

csv_path = os.path.join('Calibrated', 'GIN_calibrated.csv')
calibrated.to_csv(csv_path, index=False)

# NB : elec_current_and_future renvoie (national, RURAL, URBAIN), mais l'appel
# ci-dessus les reçoit dans l'ordre (urbain, rural). Donc :
#   urban_internal_elec_ratio contient le taux RURAL
#   rural_internal_elec_ratio contient le taux URBAIN
modelled_urban = float(rural_internal_elec_ratio)
modelled_rural = float(urban_internal_elec_ratio)

calib_info = {
    'start_year': start_year, 'intermediate_year': intermediate_year, 'end_year': end_year,
    'pop_start_year': pop_start_year, 'end_year_pop': end_year_pop,
    'urban_ratio_start_year': urban_ratio_start_year, 'urban_ratio_end_year': urban_ratio_end_year,
    'elec_ratio_start_year': elec_ratio_start_year,
    'urban_elec_ratio': urban_elec_ratio, 'rural_elec_ratio': rural_elec_ratio,
    'min_night_lights': min_night_lights, 'min_pop': min_pop,
    'max_service_transformer_distance': max_service_transformer_distance,
    'max_mv_line_distance': max_mv_line_distance, 'max_hv_line_distance': max_hv_line_distance,
    'pop_threshold_applied': thr,
    'modelled_elec_rate': float(elec_modelled),
    'modelled_urban_elec_rate': modelled_urban,     # corrigé
    'modelled_rural_elec_rate': modelled_rural,     # corrigé
}
with open(os.path.join('Calibrated', 'GIN_calibrated_parameters.json'), 'w') as f:
    json.dump(calib_info, f, indent=2)

print('Saved', os.path.abspath(csv_path), '-', len(calibrated), 'settlements,', len(calibrated.columns), 'columns')
print('Modelled electrification: total %.3f (target %.3f), urban %.3f, rural %.3f' % (
    elec_modelled, elec_ratio_start_year, modelled_urban, modelled_rural))

The figure below show the results of the calibration. Settlements in **blue** are considered to be (at least partly) electrified already in the start year of the analysis, while settlements in **grey* are yet to be electrified. Re-running the calibration step with different intial values may change the map below.

In [ ]:
from matplotlib import pyplot as plt
colors = ['#D3D3D3','#808080']
plt.figure(figsize=(12,12))
plt.plot(onsseter.df.loc[onsseter.df[SET_ELEC_CURRENT]==0, SET_X_DEG], onsseter.df.loc[onsseter.df[SET_ELEC_CURRENT]==0, SET_Y_DEG], color='#D3D3D3', marker=',', linestyle='none')
plt.plot(onsseter.df.loc[onsseter.df[SET_ELEC_CURRENT]==1, SET_X_DEG], onsseter.df.loc[onsseter.df[SET_ELEC_CURRENT]==1, SET_Y_DEG], color='#00008B', marker=',', linestyle='none')
if onsseter.df[SET_X_DEG].max() - onsseter.df[SET_X_DEG].min() > onsseter.df[SET_Y_DEG].max() - onsseter.df[SET_Y_DEG].min():
    plt.xlim(onsseter.df[SET_X_DEG].min() - 1, onsseter.df[SET_X_DEG].max() + 1)
    plt.ylim((onsseter.df[SET_Y_DEG].min()+onsseter.df[SET_Y_DEG].max())/2 - 0.5*abs(onsseter.df[SET_X_DEG].max() - onsseter.df[SET_X_DEG].min()) - 1, (onsseter.df[SET_Y_DEG].min()+onsseter.df[SET_Y_DEG].max())/2 + 0.5*abs(onsseter.df[SET_X_DEG].max() - onsseter.df[SET_X_DEG].min()) + 1)
else:
    plt.xlim((onsseter.df[SET_X_DEG].min()+onsseter.df[SET_X_DEG].max())/2 - 0.5*abs(onsseter.df[SET_Y_DEG].max() - onsseter.df[SET_Y_DEG].min()) - 1, (onsseter.df[SET_X_DEG].min()+onsseter.df[SET_X_DEG].max())/2 + 0.5*abs(onsseter.df[SET_Y_DEG].max() - onsseter.df[SET_Y_DEG].min()) + 1)
    plt.ylim(onsseter.df[SET_Y_DEG].min() -1, onsseter.df[SET_Y_DEG].max() +1)
plt.figure(figsize=(30,30))
logging.getLogger('matplotlib.font_manager').disabled = True

# 7. Run a scenario from the control file

Scenario levers (demand tiers, grid generation cost, connection caps, technology switches, costs) are no
longer typed in this notebook. They come from `Scenarios/guinea_scenarios.json`, which lists every
scenario of the study with every lever explicit, and they are executed by `gep_onsset/scenario_runner.py`,
the same code path as the command line:

    python run_scenarios.py Scenarios/guinea_scenarios.json --only BU_g0.1

The cells above (sections 1 to 5) remain the interactive calibration front end. Their lever values apply
to the calibration step only; the control file's `calibration` block must say the same.

Outputs land under `GuineaRuns/Outputs/notebook/<scenario_id>/` as the three CSVs this notebook used to
write (`_Results`, `_Summaries`, `_Variables`) plus a `manifest.json` recording every parameter, the
input file hash and the code version.


In [ ]:
from scenario_control import load_control
from scenario_runner import run_scenario, summary_from_manifest

control = load_control('Scenarios/guinea_scenarios.json')
print('Scenarios:', control.ids())

scenario_id = 'BU_g0.1'          # change to any id listed above
batch_dir = control.paths['output_root'] / 'notebook'
manifest = run_scenario(control, scenario_id, batch_dir, force=True)
summary_table = summary_from_manifest(batch_dir, scenario_id)
yearsofanalysis = [control.defaults['intermediate_year'], control.defaults['end_year']]
summary_table


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Technologies in the order of summary_table (without the Total row)
summary_plot = summary_table.drop(labels='Total', axis=0)
tech_names = summary_plot.index.tolist()
tech_colors = {'Grid_dens': '#4e53de', 'Grid_ext': '#a6aaff', 'SA_PV': '#ffc700',
               'MG_PV_Battery': '#e628a0', 'MG_PV': '#e628a0', 'MG_Hydro': '#28e66d',
               'Non-electrified': '#D3D3D3'}
bar_colors = [tech_colors.get(t, '#808080') for t in tech_names]

# The four indicators, one panel each. {y} is replaced by each year of analysis.
panels = [('Population{y}', 'Population'),
          ('NewConnections{y}', 'New connections'),
          ('Capacity{y} (MW)', 'Capacity (MW)'),
          ('Investment{y} (million USD)', 'Investment (million USD)')]

years = list(yearsofanalysis)          # e.g. [2030, 2040]
n_years = len(years)
width = 0.8 / n_years
x = np.arange(len(tech_names))
alphas = np.linspace(0.45, 1.0, n_years)        # earlier year lighter, last year solid
hatches = ['//', '\\\\', '..', ''][:n_years - 1] + ['']

fig, axarr = plt.subplots(2, 2, figsize=(18, 12))
for ax, (pattern, title) in zip(axarr.flatten(), panels):
    for i, yr in enumerate(years):
        col = pattern.format(y=yr)
        offset = (i - (n_years - 1) / 2) * width
        ax.bar(x + offset, summary_plot[col].astype(float), width=width * 0.95,
               color=bar_colors, alpha=alphas[i], hatch=hatches[i], edgecolor='white', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(tech_names, rotation=30, ha='right', fontsize=11)
    ax.set_ylabel(title, fontsize=13)
    ax.set_title(title, fontsize=14)
    ax.tick_params(axis='y', labelsize=11)

# One legend for the years (same colours, different shading)
handles = [Patch(facecolor='#808080', alpha=alphas[i], hatch=hatches[i], edgecolor='white', label=str(yr))
           for i, yr in enumerate(years)]
fig.legend(handles=handles, loc='upper right', fontsize=12, title='Year')
plt.tight_layout()
plt.show()


In [ ]:
"""
Join the cluster file with the calibrated CSV, then display the calibration
results as the model (GEP-OnSSET) will use them.

Works as a script or can be pasted into a cell of GEP_Generator.ipynb.
Edit the CONFIG section only.
"""

import os
import pandas as pd

try:
    import geopandas as gpd
except ImportError:
    gpd = None

# ----------------------------- CONFIG ---------------------------------
# Paths are relative to the project folder (gep-onsset_guinea); run from there
CSV_PATH = os.path.join("Calibrated", "GIN_calibrated.csv")
CLUSTER_PATH = os.path.join("GIN Data", "Settlements", "clusters.shp")  # needs geopandas
OUTPUT_PATH = os.path.join("Calibrated", "GIN_calibrated_with_clusters.csv")

KEY = "id"              # key column present in both files
JOIN_HOW = "left"       # keep every settlement from the CSV

# Population column to compare calibrated electrified population against.
# ElecPopCalib is on the calibration start year, so use the matching population.
POP_REF = "PopStartYear"   # falls back to Pop2025, then Pop if not found

# Calibration columns shown in the table (only those found are displayed)
CALIBRATION_COLS = [
    "id", "Admin_1", "X_deg", "Y_deg", "IsUrban", "GridCellArea",
    "Pop", "PopStartYear", "Pop2025", "NightLights",
    "ElecPop", "ElecPopCalib", "GridDistCalibElec", "ElecStart",
    "FinalElecCode2025", "InCluster", "Area", "Population", "NightLight",
]
# ----------------------------------------------------------------------


def load_table(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".gpkg", ".shp", ".geojson"):
        if gpd is None:
            raise ImportError("geopandas is required to read " + ext + " files")
        return pd.DataFrame(gpd.read_file(path).drop(columns="geometry", errors="ignore"))
    return pd.read_csv(path)


def show(df, title, n=20):
    print(f"\n=== {title} ===")
    try:
        display(df.head(n))                 # inside Jupyter
    except NameError:
        print(df.head(n).to_string())       # plain script


def main():
    data = load_table(CSV_PATH)
    clusters = load_table(CLUSTER_PATH)

    # Columns already in the CSV (Country, ElecPop, IsUrban...) are kept from the CSV
    dup = [c for c in clusters.columns if c in data.columns and c != KEY]
    clusters = clusters.drop(columns=dup)
    clusters["InCluster"] = 1

    joined = data.merge(clusters, on=KEY, how=JOIN_HOW, validate="m:1")
    joined["InCluster"] = joined["InCluster"].fillna(0).astype(int)

    # --- join diagnostics
    n_in = int(joined["InCluster"].sum())
    print(f"CSV rows      : {len(data):,}")
    print(f"Cluster rows  : {len(clusters):,}")
    print(f"Joined rows   : {len(joined):,}")
    print(f"In a cluster  : {n_in:,}  |  outside clusters: {len(joined) - n_in:,}")

    # --- calibration table
    cols = [c for c in CALIBRATION_COLS if c in joined.columns]
    calib = joined[cols]
    show(calib, "Calibration table - first 20 settlements")

    if "ElecPopCalib" in calib.columns:
        elec = calib[calib["ElecPopCalib"] > 0].sort_values("ElecPopCalib", ascending=False)
        show(elec, f"Electrified settlements ({len(elec):,}) - largest first")

    print("\n=== Summary statistics ===")
    print(calib.describe().T.to_string())

    # --- calibration totals and sanity checks
    pop_col = next((c for c in (POP_REF, "Pop2025", "Pop") if c in joined.columns), None)
    if pop_col and "ElecPopCalib" in joined.columns:
        tot_pop = joined[pop_col].sum()
        tot_el = joined["ElecPopCalib"].sum()
        print(f"\nTotal population ({pop_col}) : {tot_pop:,.0f}")
        print(f"Calibrated electrified pop   : {tot_el:,.0f}")
        print(f"Calibrated electrification   : {tot_el / tot_pop:.1%}")
        over = (joined["ElecPopCalib"] > joined[pop_col]).sum()
        print(f"Settlements with ElecPopCalib > {pop_col}: {over:,}"
              + ("  <-- check the population year used" if over else ""))

    if "FinalElecCode2025" in joined.columns:
        print("\nFinalElecCode2025 (settlements):")
        print(joined["FinalElecCode2025"].value_counts(dropna=False).sort_index().to_string())

    if "ElecPop" in joined.columns and joined["ElecPop"].sum() == 0:
        print("\nNote: ElecPop is 0 everywhere; electrification comes from ElecPopCalib.")

    joined.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSaved joined file -> {OUTPUT_PATH}")
    return joined


if __name__ == "__main__":
    df = main()

In [ ]:
"""
Electrification status map for Guinea (same layout as the Senegal map):
electrified vs non-electrified settlements, Admin2 boundaries, electrification
rate box, legend and statistics panel.

Run from the project folder (gep-onsset_guinea), or paste into a notebook cell.
Edit the CONFIG section only.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

try:
    import geopandas as gpd
except ImportError:
    gpd = None

# ----------------------------- CONFIG ---------------------------------
CSV_PATHS = [  # first one that exists is used
    os.path.join("Calibrated", "GIN_calibrated_with_clusters.csv"),
    os.path.join("Calibrated", "GIN_calibrated.csv"),
]
ADMIN2_PATH = os.path.join("GIN Data", "Admin", "gin_admin2.shp")  # change; skipped if missing
OUTPUT_PNG = os.path.join("Calibrated", "GIN_electrification_status_2025.png")

COUNTRY = "GIN"
YEAR = 2025
ELEC_CODE_COL = "FinalElecCode2025"   # 1 = electrified, 99 = not electrified
ELEC_VALUE = 1
POP_COL = "PopStartYear"              # population for the electrification rate
ELEC_POP_COL = "ElecPopCalib"
ADMIN2_NAME_COL = "Admin_2"           # in the CSV; used for the district count / fallback labels
LOCALITY_COL = None                   # optional: column identifying a locality (shows locality stats)

COL_ELEC = "#0B4F8A"
COL_NON = "#FF8A5B"
# ----------------------------------------------------------------------


def load_data():
    path = next((p for p in CSV_PATHS if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError("None of these files exist: " + ", ".join(CSV_PATHS))
    print("Reading", path)
    return pd.read_csv(path)


def panel_box(fig, rect, facecolor, edgecolor, lw=1.5):
    """Rounded box in figure coordinates."""
    ax = fig.add_axes(rect)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.add_patch(FancyBboxPatch((0.02, 0.02), 0.96, 0.96,
                                boxstyle="round,pad=0,rounding_size=0.06",
                                fc=facecolor, ec=edgecolor, lw=lw,
                                transform=ax.transAxes))
    return ax


def main():
    df = load_data()
    elec = df[ELEC_CODE_COL] == ELEC_VALUE
    n_tot, n_el = len(df), int(elec.sum())
    n_non = n_tot - n_el

    rate = df[ELEC_POP_COL].sum() / df[POP_COL].sum() * 100  # population-weighted
    n_admin2 = df[ADMIN2_NAME_COL].nunique() if ADMIN2_NAME_COL in df.columns else None

    fig = plt.figure(figsize=(16, 11))
    ax = fig.add_axes([0.04, 0.07, 0.68, 0.86])

    # --- settlements (non-electrified first so the electrified ones stay visible)
    ax.scatter(df.loc[~elec, "X_deg"], df.loc[~elec, "Y_deg"], s=3, c=COL_NON,
               alpha=0.8, linewidths=0, rasterized=True, zorder=2)
    ax.scatter(df.loc[elec, "X_deg"], df.loc[elec, "Y_deg"], s=4, c=COL_ELEC,
               alpha=0.9, linewidths=0, rasterized=True, zorder=3)

    # --- Admin2 boundaries and labels
    has_boundaries = False
    if gpd is not None and os.path.exists(ADMIN2_PATH):
        adm = gpd.read_file(ADMIN2_PATH)
        adm.boundary.plot(ax=ax, color="#777777", linewidth=0.8, zorder=4)
        has_boundaries = True
        n_admin2 = len(adm)
        name_col = next((c for c in adm.columns
                         if c.lower() in ("admin2name", "name_2", "shapename", "adm2_name", "admin_2", "name")), None)
        if name_col:
            for _, r in adm.iterrows():
                if r.geometry is None or r.geometry.is_empty:
                    continue
                pt = r.geometry.representative_point()
                ax.text(pt.x, pt.y, str(r[name_col]), fontsize=8, color="#999999",
                        style="italic", ha="center", va="center", zorder=5)
    else:
        print(f"Admin2 boundaries not found at {ADMIN2_PATH} - drawing without boundaries.")

    ax.set_aspect("equal")
    ax.set_xlabel("Longitude", fontsize=14, fontweight="bold")
    ax.set_ylabel("Latitude", fontsize=14, fontweight="bold")
    ax.grid(True, ls="--", lw=0.4, alpha=0.4)
    ax.set_title(f"{COUNTRY} Electrification Status ({YEAR})", fontsize=22, fontweight="bold", pad=14)

    # --- north arrow
    ax.annotate("N", xy=(0.04, 0.95), xytext=(0.04, 0.85), xycoords="axes fraction",
                ha="center", fontsize=18, fontweight="bold",
                arrowprops=dict(facecolor="black", width=3, headwidth=10))

    # --- right panel: rate box
    rb = panel_box(fig, [0.75, 0.76, 0.22, 0.10], "#BDE7D0", "#BDE7D0")
    rb.text(0.5, 0.5, f"Electrification Rate\n{rate:.1f}%", ha="center", va="center",
            fontsize=18, fontweight="bold")

    # --- legend
    handles = [
        Line2D([], [], marker="o", ls="", ms=14, color=COL_ELEC, label="Electrified"),
        Line2D([], [], marker="o", ls="", ms=14, color=COL_NON, label="Non-electrified"),
    ]
    if has_boundaries:
        handles.append(Line2D([], [], color="#777777", lw=2.5, label="Admin2 Boundaries"))
    leg_ax = fig.add_axes([0.75, 0.52, 0.22, 0.18])
    leg_ax.axis("off")
    leg = leg_ax.legend(handles=handles, loc="center", frameon=False, fontsize=14,
                        title="Legend", title_fontsize=20, labelspacing=1.2, handletextpad=1.2)
    leg.get_title().set_fontweight("bold")

    # --- statistics box
    lines = [
        "Statistics", "-" * 20, "",
        "Total Settlements:", f"{n_tot:,}", "",
        "Electrified:", f"{n_el:,} ({n_el / n_tot:.1%})", "",
        "Non-electrified:", f"{n_non:,} ({n_non / n_tot:.1%})",
    ]
    if LOCALITY_COL and LOCALITY_COL in df.columns:
        loc = df.groupby(LOCALITY_COL)[ELEC_CODE_COL].apply(lambda s: (s == ELEC_VALUE).any())
        lines += ["", "-" * 20, "", "Total Localities:", f"{len(loc):,}", "",
                  "Loc. Electrified:", f"{int(loc.sum()):,} ({loc.mean():.1%})", "",
                  "Loc. Non-electrified:", f"{int((~loc).sum()):,} ({(~loc).mean():.1%})"]
    if n_admin2:
        lines += ["", "Admin2 Districts:", f"{n_admin2}"]
    sb = panel_box(fig, [0.75, 0.07, 0.22, 0.42], "#F0F0F0", "#777777", lw=2)
    sb.text(0.5, 0.96, "\n".join(lines), ha="center", va="top", fontsize=11,
            family="monospace", linespacing=1.5)

    os.makedirs(os.path.dirname(OUTPUT_PNG) or ".", exist_ok=True)
    fig.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight")
    print("Saved map ->", OUTPUT_PNG)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
"""
Electrification status map for Guinea (same layout as the Senegal map):
electrified vs non-electrified settlements, Admin2 boundaries, electrification
rate box, legend and statistics panel.

Run from the project folder (gep-onsset_guinea), or paste into a notebook cell.
Edit the CONFIG section only.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

try:
    import geopandas as gpd
except ImportError:
    gpd = None

# ----------------------------- CONFIG ---------------------------------
CSV_PATHS = [  # first one that exists is used
    os.path.join("Calibrated", "GIN_calibrated_with_clusters.csv"),
    os.path.join("Calibrated", "GIN_calibrated.csv"),
]
ADMIN2_PATH = os.path.join("GIN Data", "Boundaries", "gadm41_GIN_2.shp")  # GADM 4.1 level 2 (skipped if missing)
ADMIN2_LABEL_COL = "NAME_2"           # GADM district name column (set None to hide labels)
OUTPUT_PNG = os.path.join("Calibrated", "GIN_electrification_status_2025.png")

COUNTRY = "GIN"
YEAR = 2025
ELEC_CODE_COL = "FinalElecCode2025"   # 1 = electrified, 99 = not electrified
ELEC_VALUE = 1
POP_COL = "PopStartYear"              # population for the electrification rate
ELEC_POP_COL = "ElecPopCalib"
ADMIN2_NAME_COL = "Admin_2"           # in the CSV; used for the district count / fallback labels
LOCALITY_COL = None                   # optional: column identifying a locality (shows locality stats)

COL_ELEC = "#0B4F8A"
COL_NON = "#FF8A5B"
# ----------------------------------------------------------------------


def load_data():
    path = next((p for p in CSV_PATHS if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError("None of these files exist: " + ", ".join(CSV_PATHS))
    print("Reading", path)
    return pd.read_csv(path)


def panel_box(fig, rect, facecolor, edgecolor, lw=1.5):
    """Rounded box in figure coordinates."""
    ax = fig.add_axes(rect)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.add_patch(FancyBboxPatch((0.02, 0.02), 0.96, 0.96,
                                boxstyle="round,pad=0,rounding_size=0.06",
                                fc=facecolor, ec=edgecolor, lw=lw,
                                transform=ax.transAxes))
    return ax


def main():
    df = load_data()
    elec = df[ELEC_CODE_COL] == ELEC_VALUE
    n_tot, n_el = len(df), int(elec.sum())
    n_non = n_tot - n_el

    rate = df[ELEC_POP_COL].sum() / df[POP_COL].sum() * 100  # population-weighted
    n_admin2 = df[ADMIN2_NAME_COL].nunique() if ADMIN2_NAME_COL in df.columns else None

    fig = plt.figure(figsize=(16, 11))
    ax = fig.add_axes([0.04, 0.07, 0.68, 0.86])

    # --- settlements (non-electrified first so the electrified ones stay visible)
    ax.scatter(df.loc[~elec, "X_deg"], df.loc[~elec, "Y_deg"], s=3, c=COL_NON,
               alpha=0.8, linewidths=0, rasterized=True, zorder=2)
    ax.scatter(df.loc[elec, "X_deg"], df.loc[elec, "Y_deg"], s=4, c=COL_ELEC,
               alpha=0.9, linewidths=0, rasterized=True, zorder=3)

    # --- Admin2 boundaries and labels
    has_boundaries = False
    if gpd is not None and os.path.exists(ADMIN2_PATH):
        adm = gpd.read_file(ADMIN2_PATH)
        adm.boundary.plot(ax=ax, color="#777777", linewidth=0.8, zorder=4)
        has_boundaries = True
        n_admin2 = len(adm)
        if adm.crs is not None and adm.crs.to_epsg() != 4326:
            adm = adm.to_crs(epsg=4326)   # settlements are in degrees (lon/lat)
        name_col = ADMIN2_LABEL_COL if ADMIN2_LABEL_COL in adm.columns else None
        if name_col:
            for _, r in adm.iterrows():
                if r.geometry is None or r.geometry.is_empty:
                    continue
                pt = r.geometry.representative_point()
                ax.text(pt.x, pt.y, str(r[name_col]), fontsize=8, color="#999999",
                        style="italic", ha="center", va="center", zorder=5)
    else:
        print(f"Admin2 boundaries not found at {ADMIN2_PATH} - drawing without boundaries.")

    ax.set_aspect("equal")
    ax.set_xlabel("Longitude", fontsize=14, fontweight="bold")
    ax.set_ylabel("Latitude", fontsize=14, fontweight="bold")
    ax.grid(True, ls="--", lw=0.4, alpha=0.4)
    ax.set_title(f"{COUNTRY} Electrification Status ({YEAR})", fontsize=22, fontweight="bold", pad=14)

    # --- north arrow
    ax.annotate("N", xy=(0.04, 0.95), xytext=(0.04, 0.85), xycoords="axes fraction",
                ha="center", fontsize=18, fontweight="bold",
                arrowprops=dict(facecolor="black", width=3, headwidth=10))

    # --- right panel: rate box
    rb = panel_box(fig, [0.75, 0.76, 0.22, 0.10], "#BDE7D0", "#BDE7D0")
    rb.text(0.5, 0.5, f"Electrification Rate\n{rate:.1f}%", ha="center", va="center",
            fontsize=18, fontweight="bold")

    # --- legend
    handles = [
        Line2D([], [], marker="o", ls="", ms=14, color=COL_ELEC, label="Electrified"),
        Line2D([], [], marker="o", ls="", ms=14, color=COL_NON, label="Non-electrified"),
    ]
    if has_boundaries:
        handles.append(Line2D([], [], color="#777777", lw=2.5, label="Admin2 Boundaries"))
    leg_ax = fig.add_axes([0.75, 0.52, 0.22, 0.18])
    leg_ax.axis("off")
    leg = leg_ax.legend(handles=handles, loc="center", frameon=False, fontsize=14,
                        title="Legend", title_fontsize=20, labelspacing=1.2, handletextpad=1.2)
    leg.get_title().set_fontweight("bold")

    # --- statistics box
    lines = [
        "Statistics", "-" * 20, "",
        "Total Settlements:", f"{n_tot:,}", "",
        "Electrified:", f"{n_el:,} ({n_el / n_tot:.1%})", "",
        "Non-electrified:", f"{n_non:,} ({n_non / n_tot:.1%})",
    ]
    if LOCALITY_COL and LOCALITY_COL in df.columns:
        loc = df.groupby(LOCALITY_COL)[ELEC_CODE_COL].apply(lambda s: (s == ELEC_VALUE).any())
        lines += ["", "-" * 20, "", "Total Localities:", f"{len(loc):,}", "",
                  "Loc. Electrified:", f"{int(loc.sum()):,} ({loc.mean():.1%})", "",
                  "Loc. Non-electrified:", f"{int((~loc).sum()):,} ({(~loc).mean():.1%})"]
    if n_admin2:
        lines += ["", "Admin2 Districts:", f"{n_admin2}"]
    sb = panel_box(fig, [0.75, 0.07, 0.22, 0.42], "#F0F0F0", "#777777", lw=2)
    sb.text(0.5, 0.96, "\n".join(lines), ha="center", va="top", fontsize=11,
            family="monospace", linespacing=1.5)

    os.makedirs(os.path.dirname(OUTPUT_PNG) or ".", exist_ok=True)
    fig.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight")
    print("Saved map ->", OUTPUT_PNG)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
"""
Electrification status map for Guinea (same layout as the Senegal map):
electrified vs non-electrified settlements, Admin2 boundaries, electrification
rate box, legend and statistics panel.

Run from the project folder (gep-onsset_guinea), or paste into a notebook cell.
Edit the CONFIG section only.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

try:
    import geopandas as gpd
except ImportError:
    gpd = None

# ----------------------------- CONFIG ---------------------------------
CSV_PATHS = [  # first one that exists is used
    os.path.join("Calibrated", "GIN_calibrated_with_clusters.csv"),
    os.path.join("Calibrated", "GIN_calibrated.csv"),
]
ADMIN2_PATH = os.path.join("GIN Data", "Boundaries", "gadm41_GIN_2.shp")  # GADM 4.1 level 2 (skipped if missing)
ADMIN2_LABEL_COL = "ShapeName"        # attribute shown as district label (set None to hide labels)
OUTPUT_PNG = os.path.join("Calibrated", "GIN_electrification_status_2025.png")

COUNTRY = "GIN"
YEAR = 2025
ELEC_CODE_COL = "FinalElecCode2025"   # 1 = electrified, 99 = not electrified
ELEC_VALUE = 1
POP_COL = "PopStartYear"              # population for the electrification rate
ELEC_POP_COL = "ElecPopCalib"
ADMIN2_NAME_COL = "Admin_2"           # in the CSV; used for the district count / fallback labels
LOCALITY_COL = None                   # optional: column identifying a locality (shows locality stats)

COL_ELEC = "#0B4F8A"
COL_NON = "#FF8A5B"
# ----------------------------------------------------------------------


def load_data():
    path = next((p for p in CSV_PATHS if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError("None of these files exist: " + ", ".join(CSV_PATHS))
    print("Reading", path)
    return pd.read_csv(path)


def panel_box(fig, rect, facecolor, edgecolor, lw=1.5):
    """Rounded box in figure coordinates."""
    ax = fig.add_axes(rect)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.add_patch(FancyBboxPatch((0.02, 0.02), 0.96, 0.96,
                                boxstyle="round,pad=0,rounding_size=0.06",
                                fc=facecolor, ec=edgecolor, lw=lw,
                                transform=ax.transAxes))
    return ax


def main():
    df = load_data()
    elec = df[ELEC_CODE_COL] == ELEC_VALUE
    n_tot, n_el = len(df), int(elec.sum())
    n_non = n_tot - n_el

    rate = df[ELEC_POP_COL].sum() / df[POP_COL].sum() * 100  # population-weighted
    n_admin2 = df[ADMIN2_NAME_COL].nunique() if ADMIN2_NAME_COL in df.columns else None

    fig = plt.figure(figsize=(16, 11))
    ax = fig.add_axes([0.04, 0.07, 0.68, 0.86])

    # --- settlements (non-electrified first so the electrified ones stay visible)
    ax.scatter(df.loc[~elec, "X_deg"], df.loc[~elec, "Y_deg"], s=3, c=COL_NON,
               alpha=0.8, linewidths=0, rasterized=True, zorder=2)
    ax.scatter(df.loc[elec, "X_deg"], df.loc[elec, "Y_deg"], s=4, c=COL_ELEC,
               alpha=0.9, linewidths=0, rasterized=True, zorder=3)

    # --- Admin2 boundaries and labels
    has_boundaries = False
    if gpd is not None and os.path.exists(ADMIN2_PATH):
        adm = gpd.read_file(ADMIN2_PATH)
        adm.boundary.plot(ax=ax, color="#777777", linewidth=0.8, zorder=4)
        has_boundaries = True
        n_admin2 = len(adm)
        if adm.crs is not None and adm.crs.to_epsg() != 4326:
            adm = adm.to_crs(epsg=4326)   # settlements are in degrees (lon/lat)
        name_col = None
        if ADMIN2_LABEL_COL:
            name_col = next((c for c in adm.columns if c.lower() == ADMIN2_LABEL_COL.lower()), None)
            if name_col is None:
                print(f"Column '{ADMIN2_LABEL_COL}' not found in the boundary file. "
                      f"Available columns: {list(adm.columns)}")
        if name_col:
            for _, r in adm.iterrows():
                if r.geometry is None or r.geometry.is_empty:
                    continue
                pt = r.geometry.representative_point()
                ax.text(pt.x, pt.y, str(r[name_col]), fontsize=8, color="#999999",
                        style="italic", ha="center", va="center", zorder=5)
    else:
        print(f"Admin2 boundaries not found at {ADMIN2_PATH} - drawing without boundaries.")

    ax.set_aspect("equal")
    ax.set_xlabel("Longitude", fontsize=14, fontweight="bold")
    ax.set_ylabel("Latitude", fontsize=14, fontweight="bold")
    ax.grid(True, ls="--", lw=0.4, alpha=0.4)
    ax.set_title(f"{COUNTRY} Electrification Status ({YEAR})", fontsize=22, fontweight="bold", pad=14)

    # --- north arrow
    ax.annotate("N", xy=(0.04, 0.95), xytext=(0.04, 0.85), xycoords="axes fraction",
                ha="center", fontsize=18, fontweight="bold",
                arrowprops=dict(facecolor="black", width=3, headwidth=10))

    # --- right panel: rate box
    rb = panel_box(fig, [0.75, 0.76, 0.22, 0.10], "#BDE7D0", "#BDE7D0")
    rb.text(0.5, 0.5, f"Electrification Rate\n{rate:.1f}%", ha="center", va="center",
            fontsize=18, fontweight="bold")

    # --- legend
    handles = [
        Line2D([], [], marker="o", ls="", ms=14, color=COL_ELEC, label="Electrified"),
        Line2D([], [], marker="o", ls="", ms=14, color=COL_NON, label="Non-electrified"),
    ]
    if has_boundaries:
        handles.append(Line2D([], [], color="#777777", lw=2.5, label="Admin2 Boundaries"))
    leg_ax = fig.add_axes([0.75, 0.52, 0.22, 0.18])
    leg_ax.axis("off")
    leg = leg_ax.legend(handles=handles, loc="center", frameon=False, fontsize=14,
                        title="Legend", title_fontsize=20, labelspacing=1.2, handletextpad=1.2)
    leg.get_title().set_fontweight("bold")

    # --- statistics box
    lines = [
        "Statistics", "-" * 20, "",
        "Total Settlements:", f"{n_tot:,}", "",
        "Electrified:", f"{n_el:,} ({n_el / n_tot:.1%})", "",
        "Non-electrified:", f"{n_non:,} ({n_non / n_tot:.1%})",
    ]
    if LOCALITY_COL and LOCALITY_COL in df.columns:
        loc = df.groupby(LOCALITY_COL)[ELEC_CODE_COL].apply(lambda s: (s == ELEC_VALUE).any())
        lines += ["", "-" * 20, "", "Total Localities:", f"{len(loc):,}", "",
                  "Loc. Electrified:", f"{int(loc.sum()):,} ({loc.mean():.1%})", "",
                  "Loc. Non-electrified:", f"{int((~loc).sum()):,} ({(~loc).mean():.1%})"]
    if n_admin2:
        lines += ["", "Admin2 Districts:", f"{n_admin2}"]
    sb = panel_box(fig, [0.75, 0.07, 0.22, 0.42], "#F0F0F0", "#777777", lw=2)
    sb.text(0.5, 0.96, "\n".join(lines), ha="center", va="top", fontsize=11,
            family="monospace", linespacing=1.5)

    os.makedirs(os.path.dirname(OUTPUT_PNG) or ".", exist_ok=True)
    fig.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight")
    print("Saved map ->", OUTPUT_PNG)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
df[(df.X_deg.between(-13.9, -13.5)) & (df.Y_deg.between(9.3, 9.8))][
    ["id", "X_deg", "Y_deg", "PopStartYear", "ElecPopCalib", "FinalElecCode2025"]]

In [ ]:
"""
Electrification status map for Guinea (same layout as the Senegal map):
electrified vs non-electrified settlements, Admin2 boundaries, electrification
rate box, legend and statistics panel.

Run from the project folder (gep-onsset_guinea), or paste into a notebook cell.
Edit the CONFIG section only.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

try:
    import geopandas as gpd
except ImportError:
    gpd = None

# ----------------------------- CONFIG ---------------------------------
CSV_PATHS = [  # first one that exists is used
    os.path.join("Calibrated", "GIN_calibrated_with_clusters.csv"),
    os.path.join("Calibrated", "GIN_calibrated.csv"),
]
ADMIN2_PATH = os.path.join("GIN Data", "Boundaries", "gadm41_GIN_2.shp")  # GADM 4.1 level 2 (skipped if missing)
ADMIN2_LABEL_COL = "ShapeName"        # attribute shown as district label (set None to hide labels)
OUTPUT_PNG = os.path.join("Calibrated", "GIN_electrification_status_2025.png")

COUNTRY = "GIN"
YEAR = 2025
ELEC_CODE_COL = "FinalElecCode2025"   # 1 = electrified, 99 = not electrified
ELEC_VALUE = 1
POP_COL = "PopStartYear"              # population for the electrification rate
ELEC_POP_COL = "ElecPopCalib"
ADMIN2_NAME_COL = "Admin_2"           # in the CSV; used for the district count / fallback labels
LOCALITY_COL = None                   # optional: column identifying a locality (shows locality stats)

SIZE_BY_POP = True                    # marker size grows with population
MIN_SIZE, MAX_SIZE = 3, 150           # marker area range (points^2)

COL_ELEC = "#0B4F8A"
COL_NON = "#FF8A5B"
# ----------------------------------------------------------------------


def load_data():
    path = next((p for p in CSV_PATHS if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError("None of these files exist: " + ", ".join(CSV_PATHS))
    print("Reading", path)
    return pd.read_csv(path)


def panel_box(fig, rect, facecolor, edgecolor, lw=1.5):
    """Rounded box in figure coordinates."""
    ax = fig.add_axes(rect)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.add_patch(FancyBboxPatch((0.02, 0.02), 0.96, 0.96,
                                boxstyle="round,pad=0,rounding_size=0.06",
                                fc=facecolor, ec=edgecolor, lw=lw,
                                transform=ax.transAxes))
    return ax


def main():
    df = load_data()
    elec = df[ELEC_CODE_COL] == ELEC_VALUE
    n_tot, n_el = len(df), int(elec.sum())
    n_non = n_tot - n_el

    rate = df[ELEC_POP_COL].sum() / df[POP_COL].sum() * 100  # population-weighted
    n_admin2 = df[ADMIN2_NAME_COL].nunique() if ADMIN2_NAME_COL in df.columns else None

    fig = plt.figure(figsize=(16, 11))
    ax = fig.add_axes([0.04, 0.07, 0.68, 0.86])

    # --- settlements (non-electrified first so the electrified ones stay visible)
    if SIZE_BY_POP and POP_COL in df.columns:
        # big settlements (e.g. Conakry = one huge point) must stay visible
        lp = np.log10(df[POP_COL].clip(lower=1))
        t = (lp - lp.min()) / max(lp.max() - lp.min(), 1e-9)
        sizes = MIN_SIZE + (MAX_SIZE - MIN_SIZE) * t ** 4    # power keeps small villages small
    else:
        sizes = pd.Series(4.0, index=df.index)
    ax.scatter(df.loc[~elec, "X_deg"], df.loc[~elec, "Y_deg"], s=sizes[~elec], c=COL_NON,
               alpha=0.8, linewidths=0, rasterized=True, zorder=5)
    ax.scatter(df.loc[elec, "X_deg"], df.loc[elec, "Y_deg"], s=sizes[elec], c=COL_ELEC,
               alpha=0.9, linewidths=0, rasterized=True, zorder=6)

    # --- Admin2 boundaries and labels
    has_boundaries = False
    if gpd is not None and os.path.exists(ADMIN2_PATH):
        adm = gpd.read_file(ADMIN2_PATH)
        adm.boundary.plot(ax=ax, color="#777777", linewidth=0.8, zorder=4)
        has_boundaries = True
        n_admin2 = len(adm)
        if adm.crs is not None and adm.crs.to_epsg() != 4326:
            adm = adm.to_crs(epsg=4326)   # settlements are in degrees (lon/lat)
        name_col = None
        if ADMIN2_LABEL_COL:
            name_col = next((c for c in adm.columns if c.lower() == ADMIN2_LABEL_COL.lower()), None)
            if name_col is None:
                print(f"Column '{ADMIN2_LABEL_COL}' not found in the boundary file. "
                      f"Available columns: {list(adm.columns)}")
        if name_col:
            for _, r in adm.iterrows():
                if r.geometry is None or r.geometry.is_empty:
                    continue
                pt = r.geometry.representative_point()
                ax.text(pt.x, pt.y, str(r[name_col]), fontsize=8, color="#999999",
                        style="italic", ha="center", va="center", zorder=7)
    else:
        print(f"Admin2 boundaries not found at {ADMIN2_PATH} - drawing without boundaries.")

    ax.set_aspect("equal")
    ax.set_xlabel("Longitude", fontsize=14, fontweight="bold")
    ax.set_ylabel("Latitude", fontsize=14, fontweight="bold")
    ax.grid(True, ls="--", lw=0.4, alpha=0.4)
    ax.set_title(f"{COUNTRY} Electrification Status ({YEAR})", fontsize=22, fontweight="bold", pad=14)

    # --- north arrow
    ax.annotate("N", xy=(0.04, 0.95), xytext=(0.04, 0.85), xycoords="axes fraction",
                ha="center", fontsize=18, fontweight="bold",
                arrowprops=dict(facecolor="black", width=3, headwidth=10))

    # --- right panel: rate box
    rb = panel_box(fig, [0.75, 0.76, 0.22, 0.10], "#BDE7D0", "#BDE7D0")
    rb.text(0.5, 0.5, f"Electrification Rate\n{rate:.1f}%", ha="center", va="center",
            fontsize=18, fontweight="bold")

    # --- legend
    handles = [
        Line2D([], [], marker="o", ls="", ms=14, color=COL_ELEC, label="Electrified"),
        Line2D([], [], marker="o", ls="", ms=14, color=COL_NON, label="Non-electrified"),
    ]
    if has_boundaries:
        handles.append(Line2D([], [], color="#777777", lw=2.5, label="Admin2 Boundaries"))
    leg_ax = fig.add_axes([0.75, 0.52, 0.22, 0.18])
    leg_ax.axis("off")
    leg = leg_ax.legend(handles=handles, loc="center", frameon=False, fontsize=14,
                        title="Legend", title_fontsize=20, labelspacing=1.2, handletextpad=1.2)
    leg.get_title().set_fontweight("bold")

    # --- statistics box
    lines = [
        "Statistics", "-" * 20, "",
        "Total Settlements:", f"{n_tot:,}", "",
        "Electrified:", f"{n_el:,} ({n_el / n_tot:.1%})", "",
        "Non-electrified:", f"{n_non:,} ({n_non / n_tot:.1%})",
    ]
    if LOCALITY_COL and LOCALITY_COL in df.columns:
        loc = df.groupby(LOCALITY_COL)[ELEC_CODE_COL].apply(lambda s: (s == ELEC_VALUE).any())
        lines += ["", "-" * 20, "", "Total Localities:", f"{len(loc):,}", "",
                  "Loc. Electrified:", f"{int(loc.sum()):,} ({loc.mean():.1%})", "",
                  "Loc. Non-electrified:", f"{int((~loc).sum()):,} ({(~loc).mean():.1%})"]
    if n_admin2:
        lines += ["", "Admin2 Districts:", f"{n_admin2}"]
    sb = panel_box(fig, [0.75, 0.07, 0.22, 0.42], "#F0F0F0", "#777777", lw=2)
    sb.text(0.5, 0.96, "\n".join(lines), ha="center", va="top", fontsize=11,
            family="monospace", linespacing=1.5)

    os.makedirs(os.path.dirname(OUTPUT_PNG) or ".", exist_ok=True)
    fig.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight")
    print("Saved map ->", OUTPUT_PNG)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
"""
Electrification status map for Guinea (same layout as the Senegal map):
electrified vs non-electrified settlements, Admin2 boundaries, electrification
rate box, legend and statistics panel.

Run from the project folder (gep-onsset_guinea), or paste into a notebook cell.
Edit the CONFIG section only.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

try:
    import geopandas as gpd
except ImportError:
    gpd = None

# ----------------------------- CONFIG ---------------------------------
CSV_PATHS = [  # first one that exists is used
    os.path.join("Calibrated", "GIN_calibrated_with_clusters.csv"),
    os.path.join("Calibrated", "GIN_calibrated.csv"),
]
ADMIN2_PATH = os.path.join("GIN Data", "Boundaries", "gadm41_GIN_2.shp")  # GADM 4.1 level 2 (skipped if missing)
ADMIN2_LABEL_COL = "ShapeName"        # attribute shown as district label (set None to hide labels)
OUTPUT_PNG = os.path.join("Calibrated", "GIN_electrification_status_2025.png")

COUNTRY = "GIN"
YEAR = 2025
ELEC_CODE_COL = "FinalElecCode2025"   # 1 = electrified, 99 = not electrified
ELEC_VALUE = 1
POP_COL = "PopStartYear"              # population for the electrification rate
ELEC_POP_COL = "ElecPopCalib"
ADMIN2_NAME_COL = "Admin_2"           # in the CSV; used for the district count / fallback labels
LOCALITY_COL = None                   # optional: column identifying a locality (shows locality stats)

SIZE_BY_POP = True                    # marker size grows with population
MIN_SIZE, MAX_SIZE = 3, 150           # marker area range (points^2)

COL_ELEC = "#0B4F8A"
COL_NON = "#FF8A5B"

# Style taken from the Nigeria reference map (colours sampled from the image)
BG_COLOR = "#FFFFFF"                  # page background
LAND_COLOR = "#E8EDF3"                # country fill (pale blue-grey)
TEXT_DARK = "#111520"                 # title / main text
TEXT_MUTED = "#4A5568"                # secondary text
# fonts: first one installed is used (title looks like a condensed DIN-style face)
FONT_TITLE = ["Barlow Semi Condensed", "Bahnschrift", "DIN Alternate", "Arial Narrow", "DejaVu Sans"]
FONT_BODY = ["Segoe UI", "Open Sans", "Noto Sans", "DejaVu Sans"]
# ----------------------------------------------------------------------


def load_data():
    path = next((p for p in CSV_PATHS if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError("None of these files exist: " + ", ".join(CSV_PATHS))
    print("Reading", path)
    return pd.read_csv(path)


def panel_box(fig, rect, facecolor, edgecolor, lw=1.5):
    """Rounded box in figure coordinates."""
    ax = fig.add_axes(rect)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.add_patch(FancyBboxPatch((0.02, 0.02), 0.96, 0.96,
                                boxstyle="round,pad=0,rounding_size=0.06",
                                fc=facecolor, ec=edgecolor, lw=lw,
                                transform=ax.transAxes))
    return ax


def main():
    df = load_data()
    elec = df[ELEC_CODE_COL] == ELEC_VALUE
    n_tot, n_el = len(df), int(elec.sum())
    n_non = n_tot - n_el

    rate = df[ELEC_POP_COL].sum() / df[POP_COL].sum() * 100  # population-weighted
    n_admin2 = df[ADMIN2_NAME_COL].nunique() if ADMIN2_NAME_COL in df.columns else None

    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = FONT_BODY
    fig = plt.figure(figsize=(16, 11), facecolor=BG_COLOR)
    ax = fig.add_axes([0.04, 0.07, 0.68, 0.86], facecolor=BG_COLOR)

    # --- settlements (non-electrified first so the electrified ones stay visible)
    if SIZE_BY_POP and POP_COL in df.columns:
        # big settlements (e.g. Conakry = one huge point) must stay visible
        lp = np.log10(df[POP_COL].clip(lower=1))
        t = (lp - lp.min()) / max(lp.max() - lp.min(), 1e-9)
        sizes = MIN_SIZE + (MAX_SIZE - MIN_SIZE) * t ** 4    # power keeps small villages small
    else:
        sizes = pd.Series(4.0, index=df.index)
    ax.scatter(df.loc[~elec, "X_deg"], df.loc[~elec, "Y_deg"], s=sizes[~elec], c=COL_NON,
               alpha=0.8, linewidths=0, rasterized=True, zorder=5)
    ax.scatter(df.loc[elec, "X_deg"], df.loc[elec, "Y_deg"], s=sizes[elec], c=COL_ELEC,
               alpha=0.9, linewidths=0, rasterized=True, zorder=6)

    # --- Admin2 boundaries and labels
    has_boundaries = False
    if gpd is not None and os.path.exists(ADMIN2_PATH):
        adm = gpd.read_file(ADMIN2_PATH)
        adm.plot(ax=ax, facecolor=LAND_COLOR, edgecolor="none", zorder=1)   # land fill
        adm.boundary.plot(ax=ax, color="#777777", linewidth=0.8, zorder=4)
        has_boundaries = True
        n_admin2 = len(adm)
        if adm.crs is not None and adm.crs.to_epsg() != 4326:
            adm = adm.to_crs(epsg=4326)   # settlements are in degrees (lon/lat)
        name_col = None
        if ADMIN2_LABEL_COL:
            name_col = next((c for c in adm.columns if c.lower() == ADMIN2_LABEL_COL.lower()), None)
            if name_col is None:
                print(f"Column '{ADMIN2_LABEL_COL}' not found in the boundary file. "
                      f"Available columns: {list(adm.columns)}")
        if name_col:
            for _, r in adm.iterrows():
                if r.geometry is None or r.geometry.is_empty:
                    continue
                pt = r.geometry.representative_point()
                ax.text(pt.x, pt.y, str(r[name_col]), fontsize=8, color="#999999",
                        style="italic", ha="center", va="center", zorder=7)
    else:
        print(f"Admin2 boundaries not found at {ADMIN2_PATH} - drawing without boundaries.")

    ax.set_aspect("equal")
    ax.set_xlabel("Longitude", fontsize=14, color=TEXT_MUTED)
    ax.set_ylabel("Latitude", fontsize=14, color=TEXT_MUTED)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(True, ls="--", lw=0.4, alpha=0.4)
    ax.set_title(f"{COUNTRY} Electrification Status ({YEAR})", fontsize=26, color=TEXT_DARK,
                 fontfamily=FONT_TITLE, fontweight="semibold", pad=14, loc="left")

    # --- north arrow
    ax.annotate("N", xy=(0.04, 0.95), xytext=(0.04, 0.85), xycoords="axes fraction",
                ha="center", fontsize=18, fontweight="bold", color=TEXT_DARK, fontfamily=FONT_TITLE,
                arrowprops=dict(facecolor=TEXT_DARK, width=3, headwidth=10))

    # --- right panel: rate box
    rb = panel_box(fig, [0.75, 0.76, 0.22, 0.10], "#BDE7D0", "#BDE7D0")
    rb.text(0.5, 0.5, f"Electrification Rate\n{rate:.1f}%", ha="center", va="center",
            fontsize=20, fontweight="semibold", color=TEXT_DARK, fontfamily=FONT_TITLE)

    # --- legend
    handles = [
        Line2D([], [], marker="o", ls="", ms=14, color=COL_ELEC, label="Electrified"),
        Line2D([], [], marker="o", ls="", ms=14, color=COL_NON, label="Non-electrified"),
    ]
    if has_boundaries:
        handles.append(Line2D([], [], color="#777777", lw=2.5, label="Admin2 Boundaries"))
    leg_ax = fig.add_axes([0.75, 0.52, 0.22, 0.18])
    leg_ax.axis("off")
    leg = leg_ax.legend(handles=handles, loc="center", frameon=False, fontsize=14,
                        title="Legend", title_fontsize=20, labelspacing=1.2, handletextpad=1.2)
    leg.get_title().set_fontfamily(FONT_TITLE)
    leg.get_title().set_fontweight("semibold")
    leg.get_title().set_color(TEXT_DARK)
    for t in leg.get_texts():
        t.set_color(TEXT_MUTED)

    # --- statistics box
    lines = [
        "Statistics", "-" * 20, "",
        "Total Settlements:", f"{n_tot:,}", "",
        "Electrified:", f"{n_el:,} ({n_el / n_tot:.1%})", "",
        "Non-electrified:", f"{n_non:,} ({n_non / n_tot:.1%})",
    ]
    if LOCALITY_COL and LOCALITY_COL in df.columns:
        loc = df.groupby(LOCALITY_COL)[ELEC_CODE_COL].apply(lambda s: (s == ELEC_VALUE).any())
        lines += ["", "-" * 20, "", "Total Localities:", f"{len(loc):,}", "",
                  "Loc. Electrified:", f"{int(loc.sum()):,} ({loc.mean():.1%})", "",
                  "Loc. Non-electrified:", f"{int((~loc).sum()):,} ({(~loc).mean():.1%})"]
    if n_admin2:
        lines += ["", "Admin2 Districts:", f"{n_admin2}"]
    sb = panel_box(fig, [0.75, 0.07, 0.22, 0.42], "#F0F0F0", "#777777", lw=2)
    sb.text(0.5, 0.96, "\n".join(lines), ha="center", va="top", fontsize=12,
            color=TEXT_DARK, linespacing=1.5)

    os.makedirs(os.path.dirname(OUTPUT_PNG) or ".", exist_ok=True)
    fig.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight", facecolor=BG_COLOR)
    print("Saved map ->", OUTPUT_PNG)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
"""
Electrification status map for Guinea (same layout as the Senegal map):
electrified vs non-electrified settlements, Admin2 boundaries, electrification
rate box, legend and statistics panel.

Run from the project folder (gep-onsset_guinea), or paste into a notebook cell.
Edit the CONFIG section only.
"""

import os
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

try:
    import geopandas as gpd
except ImportError:
    gpd = None

# Silence the "findfont: score(...)" lines (matplotlib font search logged at DEBUG level)
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# ----------------------------- CONFIG ---------------------------------
CSV_PATHS = [  # first one that exists is used
    os.path.join("Calibrated", "GIN_calibrated_with_clusters.csv"),
    os.path.join("Calibrated", "GIN_calibrated.csv"),
]
ADMIN2_PATH = os.path.join("GIN Data", "Boundaries", "gadm41_GIN_2.shp")  # GADM 4.1 level 2 (skipped if missing)
ADMIN2_LABEL_COL = "ShapeName"        # attribute shown as district label (set None to hide labels)
OUTPUT_PNG = os.path.join("Calibrated", "GIN_electrification_status_2025.png")

COUNTRY = "GIN"
YEAR = 2025
ELEC_CODE_COL = "FinalElecCode2025"   # 1 = electrified, 99 = not electrified
ELEC_VALUE = 1
POP_COL = "PopStartYear"              # population for the electrification rate
ELEC_POP_COL = "ElecPopCalib"
ADMIN2_NAME_COL = "Admin_2"           # in the CSV; used for the district count / fallback labels
LOCALITY_COL = None                   # optional: column identifying a locality (shows locality stats)

SIZE_BY_POP = True                    # marker size grows with population
MIN_SIZE, MAX_SIZE = 3, 150           # marker area range (points^2)

COL_ELEC = "#0B4F8A"
COL_NON = "#FF8A5B"

# Style taken from the Nigeria reference map (colours sampled from the image)
BG_COLOR = "#FFFFFF"                  # page background
LAND_COLOR = "#E8EDF3"                # country fill (pale blue-grey)
TEXT_DARK = "#111520"                 # title / main text
TEXT_MUTED = "#4A5568"                # secondary text
# fonts: first one installed is used (title looks like a condensed DIN-style face)
FONT_TITLE = ["Barlow Semi Condensed", "Bahnschrift", "DIN Alternate", "Arial Narrow", "DejaVu Sans"]
FONT_BODY = ["Segoe UI", "Open Sans", "Noto Sans", "DejaVu Sans"]
# ----------------------------------------------------------------------


def load_data():
    path = next((p for p in CSV_PATHS if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError("None of these files exist: " + ", ".join(CSV_PATHS))
    print("Reading", path)
    return pd.read_csv(path)


def panel_box(fig, rect, facecolor, edgecolor, lw=1.5):
    """Rounded box in figure coordinates."""
    ax = fig.add_axes(rect)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.add_patch(FancyBboxPatch((0.02, 0.02), 0.96, 0.96,
                                boxstyle="round,pad=0,rounding_size=0.06",
                                fc=facecolor, ec=edgecolor, lw=lw,
                                transform=ax.transAxes))
    return ax


def main():
    df = load_data()
    elec = df[ELEC_CODE_COL] == ELEC_VALUE
    n_tot, n_el = len(df), int(elec.sum())
    n_non = n_tot - n_el

    rate = df[ELEC_POP_COL].sum() / df[POP_COL].sum() * 100  # population-weighted
    n_admin2 = df[ADMIN2_NAME_COL].nunique() if ADMIN2_NAME_COL in df.columns else None

    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = FONT_BODY
    fig = plt.figure(figsize=(16, 11), facecolor=BG_COLOR)
    ax = fig.add_axes([0.04, 0.07, 0.68, 0.86], facecolor=BG_COLOR)

    # --- settlements (non-electrified first so the electrified ones stay visible)
    if SIZE_BY_POP and POP_COL in df.columns:
        # big settlements (e.g. Conakry = one huge point) must stay visible
        lp = np.log10(df[POP_COL].clip(lower=1))
        t = (lp - lp.min()) / max(lp.max() - lp.min(), 1e-9)
        sizes = MIN_SIZE + (MAX_SIZE - MIN_SIZE) * t ** 4    # power keeps small villages small
    else:
        sizes = pd.Series(4.0, index=df.index)
    ax.scatter(df.loc[~elec, "X_deg"], df.loc[~elec, "Y_deg"], s=sizes[~elec], c=COL_NON,
               alpha=0.8, linewidths=0, rasterized=True, zorder=5)
    ax.scatter(df.loc[elec, "X_deg"], df.loc[elec, "Y_deg"], s=sizes[elec], c=COL_ELEC,
               alpha=0.9, linewidths=0, rasterized=True, zorder=6)

    # --- Admin2 boundaries and labels
    has_boundaries = False
    if gpd is not None and os.path.exists(ADMIN2_PATH):
        adm = gpd.read_file(ADMIN2_PATH)
        adm.plot(ax=ax, facecolor=LAND_COLOR, edgecolor="none", zorder=1)   # land fill
        adm.boundary.plot(ax=ax, color="#777777", linewidth=0.8, zorder=4)
        has_boundaries = True
        n_admin2 = len(adm)
        if adm.crs is not None and adm.crs.to_epsg() != 4326:
            adm = adm.to_crs(epsg=4326)   # settlements are in degrees (lon/lat)
        name_col = None
        if ADMIN2_LABEL_COL:
            name_col = next((c for c in adm.columns if c.lower() == ADMIN2_LABEL_COL.lower()), None)
            if name_col is None:
                print(f"Column '{ADMIN2_LABEL_COL}' not found in the boundary file. "
                      f"Available columns: {list(adm.columns)}")
        if name_col:
            for _, r in adm.iterrows():
                if r.geometry is None or r.geometry.is_empty:
                    continue
                pt = r.geometry.representative_point()
                ax.text(pt.x, pt.y, str(r[name_col]), fontsize=8, color="#999999",
                        style="italic", ha="center", va="center", zorder=7)
    else:
        print(f"Admin2 boundaries not found at {ADMIN2_PATH} - drawing without boundaries.")

    ax.set_aspect("equal")
    ax.set_xlabel("Longitude", fontsize=14, color=TEXT_MUTED)
    ax.set_ylabel("Latitude", fontsize=14, color=TEXT_MUTED)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(True, ls="--", lw=0.4, alpha=0.4)
    ax.set_title(f"{COUNTRY} Electrification Status ({YEAR})", fontsize=26, color=TEXT_DARK,
                 fontfamily=FONT_TITLE, fontweight="semibold", pad=14, loc="left")

    # --- north arrow
    ax.annotate("N", xy=(0.04, 0.95), xytext=(0.04, 0.85), xycoords="axes fraction",
                ha="center", fontsize=18, fontweight="bold", color=TEXT_DARK, fontfamily=FONT_TITLE,
                arrowprops=dict(facecolor=TEXT_DARK, width=3, headwidth=10))

    # --- right panel: rate box
    rb = panel_box(fig, [0.75, 0.76, 0.22, 0.10], "#BDE7D0", "#BDE7D0")
    rb.text(0.5, 0.5, f"Electrification Rate\n{rate:.1f}%", ha="center", va="center",
            fontsize=20, fontweight="semibold", color=TEXT_DARK, fontfamily=FONT_TITLE)

    # --- legend
    handles = [
        Line2D([], [], marker="o", ls="", ms=14, color=COL_ELEC, label="Electrified"),
        Line2D([], [], marker="o", ls="", ms=14, color=COL_NON, label="Non-electrified"),
    ]
    if has_boundaries:
        handles.append(Line2D([], [], color="#777777", lw=2.5, label="Admin2 Boundaries"))
    leg_ax = fig.add_axes([0.75, 0.52, 0.22, 0.18])
    leg_ax.axis("off")
    leg = leg_ax.legend(handles=handles, loc="center", frameon=False, fontsize=14,
                        title="Legend", title_fontsize=20, labelspacing=1.2, handletextpad=1.2)
    leg.get_title().set_fontfamily(FONT_TITLE)
    leg.get_title().set_fontweight("semibold")
    leg.get_title().set_color(TEXT_DARK)
    for t in leg.get_texts():
        t.set_color(TEXT_MUTED)

    # --- statistics box
    lines = [
        "Statistics", "-" * 20, "",
        "Total Settlements:", f"{n_tot:,}", "",
        "Electrified:", f"{n_el:,} ({n_el / n_tot:.1%})", "",
        "Non-electrified:", f"{n_non:,} ({n_non / n_tot:.1%})",
    ]
    if LOCALITY_COL and LOCALITY_COL in df.columns:
        loc = df.groupby(LOCALITY_COL)[ELEC_CODE_COL].apply(lambda s: (s == ELEC_VALUE).any())
        lines += ["", "-" * 20, "", "Total Localities:", f"{len(loc):,}", "",
                  "Loc. Electrified:", f"{int(loc.sum()):,} ({loc.mean():.1%})", "",
                  "Loc. Non-electrified:", f"{int((~loc).sum()):,} ({(~loc).mean():.1%})"]
    if n_admin2:
        lines += ["", "Admin2 Districts:", f"{n_admin2}"]
    sb = panel_box(fig, [0.75, 0.07, 0.22, 0.42], "#F0F0F0", "#777777", lw=2)
    sb.text(0.5, 0.96, "\n".join(lines), ha="center", va="top", fontsize=12,
            color=TEXT_DARK, linespacing=1.5)

    os.makedirs(os.path.dirname(OUTPUT_PNG) or ".", exist_ok=True)
    fig.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight", facecolor=BG_COLOR)
    print("Saved map ->", OUTPUT_PNG)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
"""
Electrification status map for Guinea (same layout as the Senegal map):
electrified vs non-electrified settlements, Admin2 boundaries, electrification
rate box, legend and statistics panel.

Run from the project folder (gep-onsset_guinea), or paste into a notebook cell.
Edit the CONFIG section only.
"""

import os
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

try:
    import geopandas as gpd
except ImportError:
    gpd = None

# Silence the "findfont: score(...)" lines (matplotlib font search logged at DEBUG level)
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# ----------------------------- CONFIG ---------------------------------
CSV_PATHS = [  # first one that exists is used
    os.path.join("Calibrated", "GIN_calibrated_with_clusters.csv"),
    os.path.join("Calibrated", "GIN_calibrated.csv"),
]
ADMIN2_PATH = os.path.join("GIN Data", "Boundaries", "gadm41_GIN_2.shp")  # GADM 4.1 level 2 (skipped if missing)
ADMIN2_LABEL_COL = "ShapeName"        # attribute shown as district label (set None to hide labels)
OUTPUT_PNG = os.path.join("Calibrated", "GIN_electrification_status_2025.png")

COUNTRY = "GIN"
YEAR = 2025
ELEC_CODE_COL = "FinalElecCode2025"   # 1 = electrified, 99 = not electrified
ELEC_VALUE = 1
POP_COL = "PopStartYear"              # population for the electrification rate
ELEC_POP_COL = "ElecPopCalib"
ADMIN2_NAME_COL = "Admin_2"           # in the CSV; used for the district count / fallback labels
LOCALITY_COL = None                   # optional: column identifying a locality (shows locality stats)

SAVE_DPI = 500                        # resolution of the saved PNG
SHOW_DPI = 150                        # resolution of the figure shown in the notebook
ADMIN_LABEL_FONTSIZE = 12             # size of the district names on the map
ADMIN_LABEL_COLOR = "#4A5568"

SIZE_BY_POP = True                    # marker size grows with population
MIN_SIZE, MAX_SIZE = 3, 150           # marker area range (points^2)

COL_ELEC = "#0B4F8A"
COL_NON = "#FF8A5B"

# Style taken from the Nigeria reference map (colours sampled from the image)
BG_COLOR = "#FFFFFF"                  # page background
LAND_COLOR = "#E8EDF3"                # country fill (pale blue-grey)
TEXT_DARK = "#111520"                 # title / main text
TEXT_MUTED = "#4A5568"                # secondary text
# fonts: first one installed is used (title looks like a condensed DIN-style face)
FONT_TITLE = ["Barlow Semi Condensed", "Bahnschrift", "DIN Alternate", "Arial Narrow", "DejaVu Sans"]
FONT_BODY = ["Segoe UI", "Open Sans", "Noto Sans", "DejaVu Sans"]
# ----------------------------------------------------------------------


def load_data():
    path = next((p for p in CSV_PATHS if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError("None of these files exist: " + ", ".join(CSV_PATHS))
    print("Reading", path)
    return pd.read_csv(path)


def panel_box(fig, rect, facecolor, edgecolor, lw=1.5):
    """Rounded box in figure coordinates."""
    ax = fig.add_axes(rect)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.add_patch(FancyBboxPatch((0.02, 0.02), 0.96, 0.96,
                                boxstyle="round,pad=0,rounding_size=0.06",
                                fc=facecolor, ec=edgecolor, lw=lw,
                                transform=ax.transAxes))
    return ax


def main():
    df = load_data()
    elec = df[ELEC_CODE_COL] == ELEC_VALUE
    n_tot, n_el = len(df), int(elec.sum())
    n_non = n_tot - n_el

    rate = df[ELEC_POP_COL].sum() / df[POP_COL].sum() * 100  # population-weighted
    n_admin2 = df[ADMIN2_NAME_COL].nunique() if ADMIN2_NAME_COL in df.columns else None

    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = FONT_BODY
    fig = plt.figure(figsize=(16, 11), facecolor=BG_COLOR, dpi=SHOW_DPI)
    ax = fig.add_axes([0.04, 0.07, 0.68, 0.86], facecolor=BG_COLOR)

    # --- settlements (non-electrified first so the electrified ones stay visible)
    if SIZE_BY_POP and POP_COL in df.columns:
        # big settlements (e.g. Conakry = one huge point) must stay visible
        lp = np.log10(df[POP_COL].clip(lower=1))
        t = (lp - lp.min()) / max(lp.max() - lp.min(), 1e-9)
        sizes = MIN_SIZE + (MAX_SIZE - MIN_SIZE) * t ** 4    # power keeps small villages small
    else:
        sizes = pd.Series(4.0, index=df.index)
    ax.scatter(df.loc[~elec, "X_deg"], df.loc[~elec, "Y_deg"], s=sizes[~elec], c=COL_NON,
               alpha=0.8, linewidths=0, rasterized=True, zorder=5)
    ax.scatter(df.loc[elec, "X_deg"], df.loc[elec, "Y_deg"], s=sizes[elec], c=COL_ELEC,
               alpha=0.9, linewidths=0, rasterized=True, zorder=6)

    # --- Admin2 boundaries and labels
    has_boundaries = False
    if gpd is not None and os.path.exists(ADMIN2_PATH):
        adm = gpd.read_file(ADMIN2_PATH)
        adm.plot(ax=ax, facecolor=LAND_COLOR, edgecolor="none", zorder=1)   # land fill
        adm.boundary.plot(ax=ax, color="#777777", linewidth=0.8, zorder=4)
        has_boundaries = True
        n_admin2 = len(adm)
        if adm.crs is not None and adm.crs.to_epsg() != 4326:
            adm = adm.to_crs(epsg=4326)   # settlements are in degrees (lon/lat)
        name_col = None
        if ADMIN2_LABEL_COL:
            name_col = next((c for c in adm.columns if c.lower() == ADMIN2_LABEL_COL.lower()), None)
            if name_col is None:
                print(f"Column '{ADMIN2_LABEL_COL}' not found in the boundary file. "
                      f"Available columns: {list(adm.columns)}")
        if name_col:
            for _, r in adm.iterrows():
                if r.geometry is None or r.geometry.is_empty:
                    continue
                pt = r.geometry.representative_point()
                ax.text(pt.x, pt.y, str(r[name_col]), fontsize=ADMIN_LABEL_FONTSIZE,
                        color=ADMIN_LABEL_COLOR, fontweight="semibold", ha="center", va="center",
                        zorder=7,
                        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.85)])
    else:
        print(f"Admin2 boundaries not found at {ADMIN2_PATH} - drawing without boundaries.")

    ax.set_aspect("equal")
    ax.set_xlabel("Longitude", fontsize=14, color=TEXT_MUTED)
    ax.set_ylabel("Latitude", fontsize=14, color=TEXT_MUTED)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(True, ls="--", lw=0.4, alpha=0.4)
    ax.set_title(f"{COUNTRY} Electrification Status ({YEAR})", fontsize=26, color=TEXT_DARK,
                 fontfamily=FONT_TITLE, fontweight="semibold", pad=14, loc="left")

    # --- north arrow
    ax.annotate("N", xy=(0.04, 0.95), xytext=(0.04, 0.85), xycoords="axes fraction",
                ha="center", fontsize=18, fontweight="bold", color=TEXT_DARK, fontfamily=FONT_TITLE,
                arrowprops=dict(facecolor=TEXT_DARK, width=3, headwidth=10))

    # --- right panel: rate box
    rb = panel_box(fig, [0.75, 0.76, 0.22, 0.10], "#BDE7D0", "#BDE7D0")
    rb.text(0.5, 0.5, f"Electrification Rate\n{rate:.1f}%", ha="center", va="center",
            fontsize=20, fontweight="semibold", color=TEXT_DARK, fontfamily=FONT_TITLE)

    # --- legend
    handles = [
        Line2D([], [], marker="o", ls="", ms=14, color=COL_ELEC, label="Electrified"),
        Line2D([], [], marker="o", ls="", ms=14, color=COL_NON, label="Non-electrified"),
    ]
    if has_boundaries:
        handles.append(Line2D([], [], color="#777777", lw=2.5, label="Admin2 Boundaries"))
    leg_ax = fig.add_axes([0.75, 0.52, 0.22, 0.18])
    leg_ax.axis("off")
    leg = leg_ax.legend(handles=handles, loc="center", frameon=False, fontsize=14,
                        title="Legend", title_fontsize=20, labelspacing=1.2, handletextpad=1.2)
    leg.get_title().set_fontfamily(FONT_TITLE)
    leg.get_title().set_fontweight("semibold")
    leg.get_title().set_color(TEXT_DARK)
    for t in leg.get_texts():
        t.set_color(TEXT_MUTED)

    # --- statistics box
    lines = [
        "Statistics", "-" * 20, "",
        "Total Settlements:", f"{n_tot:,}", "",
        "Electrified:", f"{n_el:,} ({n_el / n_tot:.1%})", "",
        "Non-electrified:", f"{n_non:,} ({n_non / n_tot:.1%})",
    ]
    if LOCALITY_COL and LOCALITY_COL in df.columns:
        loc = df.groupby(LOCALITY_COL)[ELEC_CODE_COL].apply(lambda s: (s == ELEC_VALUE).any())
        lines += ["", "-" * 20, "", "Total Localities:", f"{len(loc):,}", "",
                  "Loc. Electrified:", f"{int(loc.sum()):,} ({loc.mean():.1%})", "",
                  "Loc. Non-electrified:", f"{int((~loc).sum()):,} ({(~loc).mean():.1%})"]
    if n_admin2:
        lines += ["", "Admin2 Districts:", f"{n_admin2}"]
    sb = panel_box(fig, [0.75, 0.07, 0.22, 0.42], "#F0F0F0", "#777777", lw=2)
    sb.text(0.5, 0.96, "\n".join(lines), ha="center", va="top", fontsize=12,
            color=TEXT_DARK, linespacing=1.5)

    os.makedirs(os.path.dirname(OUTPUT_PNG) or ".", exist_ok=True)
    fig.savefig(OUTPUT_PNG, dpi=SAVE_DPI, bbox_inches="tight", facecolor=BG_COLOR)
    print("Saved map ->", OUTPUT_PNG)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
"""
Electrification status map for Guinea (same layout as the Senegal map):
electrified vs non-electrified settlements, Admin2 boundaries, electrification
rate box, legend and statistics panel.

Run from the project folder (gep-onsset_guinea), or paste into a notebook cell.
Edit the CONFIG section only.
"""

import os
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

try:
    import geopandas as gpd
except ImportError:
    gpd = None

# Silence the "findfont: score(...)" lines (matplotlib font search logged at DEBUG level)
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# ----------------------------- CONFIG ---------------------------------
CSV_PATHS = [  # first one that exists is used
    os.path.join("Calibrated", "GIN_calibrated_with_clusters.csv"),
    os.path.join("Calibrated", "GIN_calibrated.csv"),
]
ADMIN2_PATH = os.path.join("GIN Data", "Boundaries", "gadm41_GIN_2.shp")  # GADM 4.1 level 2 (skipped if missing)
ADMIN2_LABEL_COL = "ShapeName"        # attribute shown as district label (set None to hide labels)
OUTPUT_PNG = os.path.join("Calibrated", "GIN_electrification_status_2025.png")

COUNTRY = "GIN"
YEAR = 2025
ELEC_CODE_COL = "FinalElecCode2025"   # 1 = electrified, 99 = not electrified
ELEC_VALUE = 1
POP_COL = "PopStartYear"              # population for the electrification rate
ELEC_POP_COL = "ElecPopCalib"
ADMIN2_NAME_COL = "Admin_2"           # in the CSV; used for the district count / fallback labels
LOCALITY_COL = None                   # optional: column identifying a locality (shows locality stats)

SAVE_DPI = 500                        # resolution of the saved PNG
SHOW_DPI = 150                        # resolution of the figure shown in the notebook
ADMIN_LABEL_FONTSIZE = 12             # size of the district names on the map
ADMIN_LABEL_COLOR = "#4A5568"

SIZE_BY_POP = True                    # marker size grows with population
MIN_SIZE, MAX_SIZE = 3, 150           # marker area range (points^2)

COL_ELEC = "#0B4F8A"
COL_NON = "#FF8A5B"

# Style taken from the Nigeria reference map (colours sampled from the image)
BG_COLOR = "#FFFFFF"                  # page background
LAND_COLOR = "#E8EDF3"                # country fill (pale blue-grey)
TEXT_DARK = "#111520"                 # title / main text
TEXT_MUTED = "#4A5568"                # secondary text
# fonts: first one installed is used (title looks like a condensed DIN-style face)
FONT_TITLE = ["Barlow Semi Condensed", "Bahnschrift", "DIN Alternate", "Arial Narrow", "DejaVu Sans"]
FONT_BODY = ["Segoe UI", "Open Sans", "Noto Sans", "DejaVu Sans"]
# ----------------------------------------------------------------------


def load_data():
    path = next((p for p in CSV_PATHS if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError("None of these files exist: " + ", ".join(CSV_PATHS))
    print("Reading", path)
    return pd.read_csv(path)


def panel_box(fig, rect, facecolor, edgecolor, lw=1.5, pad=0.02):
    """Rounded box in figure coordinates (pad=0: the box fills the rectangle exactly)."""
    ax = fig.add_axes(rect)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    patch = FancyBboxPatch((pad, pad), 1 - 2 * pad, 1 - 2 * pad,
                           boxstyle="round,pad=0,rounding_size=0.06",
                           fc=facecolor, ec=edgecolor, lw=lw,
                           transform=ax.transAxes, clip_on=False)
    ax.add_patch(patch)
    return ax


def main():
    df = load_data()
    elec = df[ELEC_CODE_COL] == ELEC_VALUE
    n_tot, n_el = len(df), int(elec.sum())
    n_non = n_tot - n_el

    rate = df[ELEC_POP_COL].sum() / df[POP_COL].sum() * 100  # population-weighted
    n_admin2 = df[ADMIN2_NAME_COL].nunique() if ADMIN2_NAME_COL in df.columns else None

    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = FONT_BODY
    fig = plt.figure(figsize=(16, 11), facecolor=BG_COLOR, dpi=SHOW_DPI)
    ax = fig.add_axes([0.04, 0.07, 0.68, 0.86], facecolor=BG_COLOR)

    # --- settlements (non-electrified first so the electrified ones stay visible)
    if SIZE_BY_POP and POP_COL in df.columns:
        # big settlements (e.g. Conakry = one huge point) must stay visible
        lp = np.log10(df[POP_COL].clip(lower=1))
        t = (lp - lp.min()) / max(lp.max() - lp.min(), 1e-9)
        sizes = MIN_SIZE + (MAX_SIZE - MIN_SIZE) * t ** 4    # power keeps small villages small
    else:
        sizes = pd.Series(4.0, index=df.index)
    ax.scatter(df.loc[~elec, "X_deg"], df.loc[~elec, "Y_deg"], s=sizes[~elec], c=COL_NON,
               alpha=0.8, linewidths=0, rasterized=True, zorder=5)
    ax.scatter(df.loc[elec, "X_deg"], df.loc[elec, "Y_deg"], s=sizes[elec], c=COL_ELEC,
               alpha=0.9, linewidths=0, rasterized=True, zorder=6)

    # --- Admin2 boundaries and labels
    has_boundaries = False
    if gpd is not None and os.path.exists(ADMIN2_PATH):
        adm = gpd.read_file(ADMIN2_PATH)
        adm.plot(ax=ax, facecolor=LAND_COLOR, edgecolor="none", zorder=1)   # land fill
        adm.boundary.plot(ax=ax, color="#777777", linewidth=0.8, zorder=4)
        has_boundaries = True
        n_admin2 = len(adm)
        if adm.crs is not None and adm.crs.to_epsg() != 4326:
            adm = adm.to_crs(epsg=4326)   # settlements are in degrees (lon/lat)
        name_col = None
        if ADMIN2_LABEL_COL:
            name_col = next((c for c in adm.columns if c.lower() == ADMIN2_LABEL_COL.lower()), None)
            if name_col is None:
                print(f"Column '{ADMIN2_LABEL_COL}' not found in the boundary file. "
                      f"Available columns: {list(adm.columns)}")
        if name_col:
            for _, r in adm.iterrows():
                if r.geometry is None or r.geometry.is_empty:
                    continue
                pt = r.geometry.representative_point()
                ax.text(pt.x, pt.y, str(r[name_col]), fontsize=ADMIN_LABEL_FONTSIZE,
                        color=ADMIN_LABEL_COLOR, fontweight="semibold", ha="center", va="center",
                        zorder=7,
                        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.85)])
    else:
        print(f"Admin2 boundaries not found at {ADMIN2_PATH} - drawing without boundaries.")

    ax.set_aspect("equal")
    ax.set_xlabel("Longitude", fontsize=14, color=TEXT_MUTED)
    ax.set_ylabel("Latitude", fontsize=14, color=TEXT_MUTED)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(True, ls="--", lw=0.4, alpha=0.4)
    ax.set_title(f"{COUNTRY} Electrification Status ({YEAR})", fontsize=26, color=TEXT_DARK,
                 fontfamily=FONT_TITLE, fontweight="semibold", pad=14, loc="left")

    # --- north arrow
    ax.annotate("N", xy=(0.04, 0.95), xytext=(0.04, 0.85), xycoords="axes fraction",
                ha="center", fontsize=18, fontweight="bold", color=TEXT_DARK, fontfamily=FONT_TITLE,
                arrowprops=dict(facecolor=TEXT_DARK, width=3, headwidth=10))

    # --- right panel: rate box
    rb = panel_box(fig, [0.75, 0.76, 0.22, 0.10], "#BDE7D0", "#BDE7D0")
    rb.text(0.5, 0.5, f"Electrification Rate\n{rate:.1f}%", ha="center", va="center",
            fontsize=20, fontweight="semibold", color=TEXT_DARK, fontfamily=FONT_TITLE)

    # --- legend
    handles = [
        Line2D([], [], marker="o", ls="", ms=14, color=COL_ELEC, label="Electrified"),
        Line2D([], [], marker="o", ls="", ms=14, color=COL_NON, label="Non-electrified"),
    ]
    if has_boundaries:
        handles.append(Line2D([], [], color="#777777", lw=2.5, label="Admin2 Boundaries"))
    leg_ax = fig.add_axes([0.75, 0.52, 0.22, 0.18])
    leg_ax.axis("off")
    leg = leg_ax.legend(handles=handles, loc="center", frameon=False, fontsize=14,
                        title="Legend", title_fontsize=20, labelspacing=1.2, handletextpad=1.2)
    leg.get_title().set_fontfamily(FONT_TITLE)
    leg.get_title().set_fontweight("semibold")
    leg.get_title().set_color(TEXT_DARK)
    for t in leg.get_texts():
        t.set_color(TEXT_MUTED)

    # --- statistics box
    lines = [
        "Statistics", "-" * 20, "",
        "Total Settlements:", f"{n_tot:,}", "",
        "Electrified:", f"{n_el:,} ({n_el / n_tot:.1%})", "",
        "Non-electrified:", f"{n_non:,} ({n_non / n_tot:.1%})",
    ]
    if LOCALITY_COL and LOCALITY_COL in df.columns:
        loc = df.groupby(LOCALITY_COL)[ELEC_CODE_COL].apply(lambda s: (s == ELEC_VALUE).any())
        lines += ["", "-" * 20, "", "Total Localities:", f"{len(loc):,}", "",
                  "Loc. Electrified:", f"{int(loc.sum()):,} ({loc.mean():.1%})", "",
                  "Loc. Non-electrified:", f"{int((~loc).sum()):,} ({(~loc).mean():.1%})"]
    if n_admin2:
        lines += ["", "Admin2 Districts:", f"{n_admin2}"]
    # bottom of the box = bottom of the map frame (equal aspect makes the frame shorter
    # than the axes rectangle, so read its real position)
    ax.apply_aspect()
    map_bottom = ax.get_position().y0
    box_top = 0.49
    sb = panel_box(fig, [0.75, map_bottom, 0.22, box_top - map_bottom], "#F0F0F0", "#777777",
                   lw=2, pad=0)
    sb.text(0.5, 0.96, "\n".join(lines), ha="center", va="top", fontsize=12,
            color=TEXT_DARK, linespacing=1.5)

    os.makedirs(os.path.dirname(OUTPUT_PNG) or ".", exist_ok=True)
    fig.savefig(OUTPUT_PNG, dpi=SAVE_DPI, bbox_inches="tight", facecolor=BG_COLOR)
    print("Saved map ->", OUTPUT_PNG)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
"""
Electrification status map for Guinea (same layout as the Senegal map):
electrified vs non-electrified settlements, Admin2 boundaries, electrification
rate box, legend and statistics panel.

Run from the project folder (gep-onsset_guinea), or paste into a notebook cell.
Edit the CONFIG section only.
"""

import os
import time
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch

try:
    import geopandas as gpd
except ImportError:
    gpd = None

# Silence the "findfont: score(...)" lines (matplotlib font search logged at DEBUG level)
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# ----------------------------- CONFIG ---------------------------------
CSV_PATHS = [  # first one that exists is used
    os.path.join("Calibrated", "GIN_calibrated_with_clusters.csv"),
    os.path.join("Calibrated", "GIN_calibrated.csv"),
]
ADMIN2_PATH = os.path.join("GIN Data", "Boundaries", "gadm41_GIN_2.shp")  # GADM 4.1 level 2 (skipped if missing)
ADMIN2_LABEL_COL = "ShapeName"        # attribute shown as district label (set None to hide labels)
OUTPUT_PNG = os.path.join("Calibrated", "GIN_electrification_status_2025.png")

COUNTRY = "GIN"
YEAR = 2025
ELEC_CODE_COL = "FinalElecCode2025"   # 1 = electrified, 99 = not electrified
ELEC_VALUE = 1
POP_COL = "PopStartYear"              # population for the electrification rate
ELEC_POP_COL = "ElecPopCalib"
ADMIN2_NAME_COL = "Admin_2"           # in the CSV; used for the district count / fallback labels
LOCALITY_COL = None                   # optional: column identifying a locality (shows locality stats)

SAVE_DPI = 500                        # resolution of the saved PNG
SHOW_DPI = 150                        # resolution of the figure shown in the notebook
ADMIN_LABEL_FONTSIZE = 12             # size of the district names on the map
ADMIN_LABEL_COLOR = "#4A5568"

SIZE_BY_POP = True                    # marker size grows with population
MIN_SIZE, MAX_SIZE = 3, 150           # marker area range (points^2)

COL_ELEC = "#0B4F8A"
COL_NON = "#FF8A5B"

# Style taken from the Nigeria reference map (colours sampled from the image)
BG_COLOR = "#FFFFFF"                  # page background
LAND_COLOR = "#E8EDF3"                # country fill (pale blue-grey)
TEXT_DARK = "#111520"                 # title / main text
TEXT_MUTED = "#4A5568"                # secondary text
# fonts: first one installed is used (title looks like a condensed DIN-style face)
FONT_TITLE = ["Barlow Semi Condensed", "Bahnschrift", "DIN Alternate", "Arial Narrow", "DejaVu Sans"]
FONT_BODY = ["Segoe UI", "Open Sans", "Noto Sans", "DejaVu Sans"]
# ----------------------------------------------------------------------


def load_data():
    path = next((p for p in CSV_PATHS if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError("None of these files exist: " + ", ".join(CSV_PATHS))
    print("Reading", path)
    return pd.read_csv(path)


def panel_box(fig, rect, facecolor, edgecolor, lw=1.5, pad=0.02):
    """Rounded box in figure coordinates (pad=0: the box fills the rectangle exactly)."""
    ax = fig.add_axes(rect)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    patch = FancyBboxPatch((pad, pad), 1 - 2 * pad, 1 - 2 * pad,
                           boxstyle="round,pad=0,rounding_size=0.06",
                           fc=facecolor, ec=edgecolor, lw=lw,
                           transform=ax.transAxes, clip_on=False)
    ax.add_patch(patch)
    return ax


def save_figure(fig, path):
    """Save the PNG; if Windows refuses the file (locked, synced, bad path), try other names."""
    path = os.path.abspath(path)
    stem, ext = os.path.splitext(path)
    candidates = [
        path,
        f"{stem}_{time.strftime('%H%M%S')}{ext}",                       # new name, same folder
        os.path.join(os.path.expanduser("~"), os.path.basename(path)),  # user folder
    ]
    for p in candidates:
        try:
            os.makedirs(os.path.dirname(p), exist_ok=True)
            fig.savefig(p, dpi=SAVE_DPI, bbox_inches="tight", facecolor=BG_COLOR)
            print("Saved map ->", p)
            return p
        except OSError as e:
            print(f"Could not save to {p}: {e}")
    print("The map could not be saved; it is still displayed.")
    return None


def main():
    df = load_data()
    elec = df[ELEC_CODE_COL] == ELEC_VALUE
    n_tot, n_el = len(df), int(elec.sum())
    n_non = n_tot - n_el

    rate = df[ELEC_POP_COL].sum() / df[POP_COL].sum() * 100  # population-weighted
    n_admin2 = df[ADMIN2_NAME_COL].nunique() if ADMIN2_NAME_COL in df.columns else None

    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = FONT_BODY
    fig = plt.figure(figsize=(16, 11), facecolor=BG_COLOR, dpi=SHOW_DPI)
    ax = fig.add_axes([0.04, 0.07, 0.68, 0.86], facecolor=BG_COLOR)

    # --- settlements (non-electrified first so the electrified ones stay visible)
    if SIZE_BY_POP and POP_COL in df.columns:
        # big settlements (e.g. Conakry = one huge point) must stay visible
        lp = np.log10(df[POP_COL].clip(lower=1))
        t = (lp - lp.min()) / max(lp.max() - lp.min(), 1e-9)
        sizes = MIN_SIZE + (MAX_SIZE - MIN_SIZE) * t ** 4    # power keeps small villages small
    else:
        sizes = pd.Series(4.0, index=df.index)
    ax.scatter(df.loc[~elec, "X_deg"], df.loc[~elec, "Y_deg"], s=sizes[~elec], c=COL_NON,
               alpha=0.8, linewidths=0, rasterized=True, zorder=5)
    ax.scatter(df.loc[elec, "X_deg"], df.loc[elec, "Y_deg"], s=sizes[elec], c=COL_ELEC,
               alpha=0.9, linewidths=0, rasterized=True, zorder=6)

    # --- Admin2 boundaries and labels
    has_boundaries = False
    if gpd is not None and os.path.exists(ADMIN2_PATH):
        adm = gpd.read_file(ADMIN2_PATH)
        adm.plot(ax=ax, facecolor=LAND_COLOR, edgecolor="none", zorder=1)   # land fill
        adm.boundary.plot(ax=ax, color="#777777", linewidth=0.8, zorder=4)
        has_boundaries = True
        n_admin2 = len(adm)
        if adm.crs is not None and adm.crs.to_epsg() != 4326:
            adm = adm.to_crs(epsg=4326)   # settlements are in degrees (lon/lat)
        name_col = None
        if ADMIN2_LABEL_COL:
            name_col = next((c for c in adm.columns if c.lower() == ADMIN2_LABEL_COL.lower()), None)
            if name_col is None:
                print(f"Column '{ADMIN2_LABEL_COL}' not found in the boundary file. "
                      f"Available columns: {list(adm.columns)}")
        if name_col:
            for _, r in adm.iterrows():
                if r.geometry is None or r.geometry.is_empty:
                    continue
                pt = r.geometry.representative_point()
                ax.text(pt.x, pt.y, str(r[name_col]), fontsize=ADMIN_LABEL_FONTSIZE,
                        color=ADMIN_LABEL_COLOR, fontweight="semibold", ha="center", va="center",
                        zorder=7,
                        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.85)])
    else:
        print(f"Admin2 boundaries not found at {ADMIN2_PATH} - drawing without boundaries.")

    ax.set_aspect("equal")
    ax.set_xlabel("Longitude", fontsize=14, color=TEXT_MUTED)
    ax.set_ylabel("Latitude", fontsize=14, color=TEXT_MUTED)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(True, ls="--", lw=0.4, alpha=0.4)
    ax.set_title(f"{COUNTRY} Electrification Status ({YEAR})", fontsize=26, color=TEXT_DARK,
                 fontfamily=FONT_TITLE, fontweight="semibold", pad=14, loc="left")

    # --- north arrow
    ax.annotate("N", xy=(0.04, 0.95), xytext=(0.04, 0.85), xycoords="axes fraction",
                ha="center", fontsize=18, fontweight="bold", color=TEXT_DARK, fontfamily=FONT_TITLE,
                arrowprops=dict(facecolor=TEXT_DARK, width=3, headwidth=10))

    # --- right panel: rate box
    rb = panel_box(fig, [0.75, 0.76, 0.22, 0.10], "#BDE7D0", "#BDE7D0")
    rb.text(0.5, 0.5, f"Electrification Rate\n{rate:.1f}%", ha="center", va="center",
            fontsize=20, fontweight="semibold", color=TEXT_DARK, fontfamily=FONT_TITLE)

    # --- legend
    handles = [
        Line2D([], [], marker="o", ls="", ms=14, color=COL_ELEC, label="Electrified"),
        Line2D([], [], marker="o", ls="", ms=14, color=COL_NON, label="Non-electrified"),
    ]
    if has_boundaries:
        handles.append(Line2D([], [], color="#777777", lw=2.5, label="Admin2 Boundaries"))
    leg_ax = fig.add_axes([0.75, 0.52, 0.22, 0.18])
    leg_ax.axis("off")
    leg = leg_ax.legend(handles=handles, loc="center", frameon=False, fontsize=14,
                        title="Legend", title_fontsize=20, labelspacing=1.2, handletextpad=1.2)
    leg.get_title().set_fontfamily(FONT_TITLE)
    leg.get_title().set_fontweight("semibold")
    leg.get_title().set_color(TEXT_DARK)
    for t in leg.get_texts():
        t.set_color(TEXT_MUTED)

    # --- statistics box
    lines = [
        "Statistics", "-" * 20, "",
        "Total Settlements:", f"{n_tot:,}", "",
        "Electrified:", f"{n_el:,} ({n_el / n_tot:.1%})", "",
        "Non-electrified:", f"{n_non:,} ({n_non / n_tot:.1%})",
    ]
    if LOCALITY_COL and LOCALITY_COL in df.columns:
        loc = df.groupby(LOCALITY_COL)[ELEC_CODE_COL].apply(lambda s: (s == ELEC_VALUE).any())
        lines += ["", "-" * 20, "", "Total Localities:", f"{len(loc):,}", "",
                  "Loc. Electrified:", f"{int(loc.sum()):,} ({loc.mean():.1%})", "",
                  "Loc. Non-electrified:", f"{int((~loc).sum()):,} ({(~loc).mean():.1%})"]
    if n_admin2:
        lines += ["", "Admin2 Districts:", f"{n_admin2}"]
    # bottom of the box = bottom of the map frame (equal aspect makes the frame shorter
    # than the axes rectangle, so read its real position)
    ax.apply_aspect()
    map_bottom = ax.get_position().y0
    box_top = 0.49
    sb = panel_box(fig, [0.75, map_bottom, 0.22, box_top - map_bottom], "#F0F0F0", "#777777",
                   lw=2, pad=0)
    sb.text(0.5, 0.96, "\n".join(lines), ha="center", va="top", fontsize=12,
            color=TEXT_DARK, linespacing=1.5)

    save_figure(fig, OUTPUT_PNG)
    plt.show()


if __name__ == "__main__":
    main()